# CKD Patient Stratification

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
import random
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import re

from scipy import stats
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu, f_oneway
from scipy.stats.mstats import winsorize
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import KMeans, DBSCAN, MiniBatchKMeans, SpectralClustering, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score, confusion_matrix,
    classification_report, pairwise_distances
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.manifold import TSNE

import umap
import shap
import joblib

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

print('All imports loaded successfully.')

All imports loaded successfully.


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

BASE_DIR = Path.cwd()
DRIVE_OUT = str(BASE_DIR / 'astar_output') + '/'
FIG_DIR = str(BASE_DIR / 'figures') + '/'
TABLE_DIR = str(BASE_DIR / 'tables') + '/'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

PALETTE = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

print('='*70)
print('CKD PATIENT STRATIFICATION — A*-QUALITY IMPROVED PIPELINE v4')
print('='*70)
print(f'Seed: {SEED}')
print(f'Output: {DRIVE_OUT}')

CKD PATIENT STRATIFICATION — A*-QUALITY IMPROVED PIPELINE v4
Seed: 42
Output: /home/eatl/CKD PROJECT/astar_output/


## 1. Load Data

In [ ]:
FILE_PATH = str(BASE_DIR / 'Dataset' / 'ckd_primary_data.csv')
df_raw = pd.read_csv(FILE_PATH, low_memory=False)

print(f'Rows: {df_raw.shape[0]:,}   Columns: {df_raw.shape[1]}')
print(f'Memory: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

missing = pd.DataFrame({
    'Missing_N': df_raw.isnull().sum(),
    'Missing_%': (df_raw.isnull().mean() * 100).round(1),
    'Dtype': df_raw.dtypes
}).sort_values('Missing_%', ascending=False)
print(f'Columns with >50% missing: {(missing["Missing_%"] > 50).sum()}')
print(f'Columns with >95% missing: {(missing["Missing_%"] > 95).sum()}')

Rows: 115,995   Columns: 57
Memory: 291.8 MB
Columns with >50% missing: 42
Columns with >95% missing: 14


## 2. PHI De-identification

In [ ]:
phi_cols = [c for c in ['PATIENT_NAME', 'BLOOD_GROUP', 'M_STATUS_DATA'] if c in df_raw.columns]
if phi_cols:
    df_raw.drop(columns=phi_cols, inplace=True)
    print(f'Dropped {len(phi_cols)} PHI columns: {phi_cols}')
else:
    print('No PHI columns found.')

Dropped 3 PHI columns: ['PATIENT_NAME', 'BLOOD_GROUP', 'M_STATUS_DATA']


## 3. Clinical-Aware Data Cleaning

In [ ]:
df = df_raw.copy()
null_counts_before = df.isnull().sum()

def clean_age(x):
    if pd.isna(x): return np.nan
    m = re.search(r'(\d+)\s*[Yy]', str(x))
    if m: return int(m.group(1))
    m = re.search(r'^(\d+)$', str(x).strip())
    if m: return int(m.group(1))
    return np.nan

df['AGE'] = df['AGE'].apply(clean_age)
df.loc[~df['AGE'].between(1, 120), 'AGE'] = np.nan

df['WEIGHT'] = pd.to_numeric(
    df['WEIGHT'].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df.loc[~df['WEIGHT'].between(20, 300), 'WEIGHT'] = np.nan

bp = df['BLOOD_PRESSURE'].astype(str)
df['SYS_BP'] = pd.to_numeric(bp.str.extract(r'(\d+)/')[0], errors='coerce')
df['DIA_BP'] = pd.to_numeric(bp.str.extract(r'/(\d+)')[0], errors='coerce')
df.drop(columns=['BLOOD_PRESSURE'], inplace=True)
df.loc[~df['SYS_BP'].between(60, 300), 'SYS_BP'] = np.nan
df.loc[~df['DIA_BP'].between(30, 200), 'DIA_BP'] = np.nan

LAB_NUMERIC = [
    'CREATININE_RESULT', 'ALBUMIN_RESULT', 'GLUCOSE_R_RESULT',
    'HAEMOGLOBIN_RESULT', 'POTASSIUM_RESULT', 'SODIUM_RESULT',
    'CALCIUM_RESULT', 'PHOSPHOROUS_RESULT', 'EGFR_RESULT',
    'TLC_RESULT', 'RBC_COUNT_RESULT'
]

CLINICAL_RANGES = {
    'CREATININE_RESULT':  (0.1, 50.0), 'ALBUMIN_RESULT': (0.5, 8.0),
    'GLUCOSE_R_RESULT':   (1.0, 100.0), 'HAEMOGLOBIN_RESULT': (1.0, 25.0),
    'POTASSIUM_RESULT':   (1.0, 10.0), 'SODIUM_RESULT': (100, 180),
    'CALCIUM_RESULT':     (3.0, 20.0), 'PHOSPHOROUS_RESULT': (0.5, 20.0),
    'EGFR_RESULT':        (1.0, 200), 'TLC_RESULT': (0.5, 100),
    'RBC_COUNT_RESULT':   (0.5, 10.0),
}

for col in LAB_NUMERIC:
    if col not in df.columns: continue
    df[col] = pd.to_numeric(df[col].astype(str).str.extract(r'([-+]?\d*\.?\d+)')[0], errors='coerce')
    if col in CLINICAL_RANGES:
        lo, hi = CLINICAL_RANGES[col]
        df.loc[~df[col].between(lo, hi), col] = np.nan

df['GENDER'] = df['GENDER'].astype(str).str.strip().str.title()
df['GENDER'] = df['GENDER'].where(df['GENDER'].isin(['Male', 'Female']), np.nan)
df['GENDER_NUM'] = df['GENDER'].map({'Male': 1, 'Female': 0})

null_counts_after = df.isnull().sum()
nulls_added = (null_counts_after - null_counts_before)[(null_counts_after - null_counts_before) > 0]
print(f'Cleaning complete. Shape: {df.shape}')
print(f'Columns nulled by clinical filters: {len(nulls_added)}')

Cleaning complete. Shape: (115995, 56)
Columns nulled by clinical filters: 14


## 4. Drop High-Missingness Columns (>95%)

In [ ]:
miss_ratio = df.isnull().mean()
dropped_cols = miss_ratio[miss_ratio >= 0.95].index.tolist()
df = df[miss_ratio[miss_ratio < 0.95].index].copy()
print(f'Dropped {len(dropped_cols)} columns (>95% missing)')
print(f'Shape: {df.shape}')

Dropped 14 columns (>95% missing)
Shape: (115995, 42)


## FIX-1: Patient Deduplication

**Justification:** The raw data contains repeated rows for the same patient (e.g., different lab visits). Treating these as independent violates the independence assumption and inflates sample sizes.

**Note:** We do NOT perform proxy-based deduplication (e.g., by AGE+GENDER) because AGE+GENDER is not a unique patient identifier.

In [ ]:
n_before = len(df)
duplicated_rows = df.duplicated(keep='first').sum()
print(f'Exact duplicate rows: {duplicated_rows:,} / {n_before:,}')

if duplicated_rows > 0:
    df = df.drop_duplicates(keep='first').copy()
    print(f'After deduplication: {len(df):,} rows (removed {n_before - len(df):,})')
else:
    print('No exact duplicates found.')

Exact duplicate rows: 188 / 115,995
After deduplication: 115,807 rows (removed 188)


## FIX-2: Dual-Cohort Construction (AND logic for Cohort A)

**Justification:** v3 used OR logic, including patients with only Haemoglobin data in the CKD cohort. A patient without eGFR or Creatinine cannot be clinically classified as CKD. Using AND ensures the cohort is clinically meaningful.

In [ ]:
egfr_exists = 'EGFR_RESULT' in df.columns
creatinine_exists = 'CREATININE_RESULT' in df.columns
haemoglobin_exists = 'HAEMOGLOBIN_RESULT' in df.columns

# FIX-2: AND logic
if egfr_exists and creatinine_exists:
    mask_ckd = df['EGFR_RESULT'].notna() & df['CREATININE_RESULT'].notna()
elif egfr_exists:
    mask_ckd = df['EGFR_RESULT'].notna()
elif creatinine_exists:
    mask_ckd = df['CREATININE_RESULT'].notna()
else:
    mask_ckd = pd.Series(False, index=df.index)

clinical_markers = ['HAEMOGLOBIN_RESULT', 'ALBUMIN_RESULT', 'SYS_BP']
available_markers = [c for c in clinical_markers if c in df.columns]
if available_markers:
    has_clinical = df[available_markers].notna().any(axis=1)
    mask_ckd = mask_ckd & has_clinical

df_ckd_raw = df[mask_ckd].copy()

CKD_FEATURES_ORIGINAL = [
    'EGFR_RESULT','CREATININE_RESULT','HAEMOGLOBIN_RESULT',
    'ALBUMIN_RESULT','POTASSIUM_RESULT','SODIUM_RESULT',
    'CALCIUM_RESULT','AGE','SYS_BP','DIA_BP','WEIGHT','GENDER_NUM'
]
CKD_FEATURES_ORIGINAL = [c for c in CKD_FEATURES_ORIGINAL if c in df_ckd_raw.columns]

mask_hb = df['HAEMOGLOBIN_RESULT'].notna()
df_hb_raw = df[mask_hb].copy()

HB_FEATURES_ORIGINAL = [
    'HAEMOGLOBIN_RESULT','TLC_RESULT','RBC_COUNT_RESULT',
    'AGE','SYS_BP','DIA_BP','WEIGHT','GENDER_NUM'
]
HB_FEATURES_ORIGINAL = [c for c in HB_FEATURES_ORIGINAL if c in df_hb_raw.columns]

print(f'Cohort A (CKD, AND logic): {len(df_ckd_raw):>7,} patients  |  {len(CKD_FEATURES_ORIGINAL)} features')
print(f'Cohort B (Haemoglobin):     {len(df_hb_raw):>7,} patients  |  {len(HB_FEATURES_ORIGINAL)} features')
print(f'Features A: {CKD_FEATURES_ORIGINAL}')

Cohort A (CKD, AND logic):   8,292 patients  |  12 features
Cohort B (Haemoglobin):      94,688 patients  |  8 features
Features A: ['EGFR_RESULT', 'CREATININE_RESULT', 'HAEMOGLOBIN_RESULT', 'ALBUMIN_RESULT', 'POTASSIUM_RESULT', 'SODIUM_RESULT', 'CALCIUM_RESULT', 'AGE', 'SYS_BP', 'DIA_BP', 'WEIGHT', 'GENDER_NUM']


## 5. Train/Validation Split (stratified by gender)

**Justification:** Simple random split may create imbalanced cohorts. Stratifying by gender ensures both splits have similar demographics.

In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_FRAC = 0.70
VAL_FRAC = 0.30

df_ckd_raw['_strat'] = df_ckd_raw['GENDER_NUM'].fillna(0.5).astype(str)
df_hb_raw['_strat'] = df_hb_raw['GENDER_NUM'].fillna(0.5).astype(str)

train_idx_a, val_idx_a = train_test_split(
    df_ckd_raw.index, test_size=VAL_FRAC, random_state=SEED,
    stratify=df_ckd_raw['_strat'], shuffle=True)
train_idx_b, val_idx_b = train_test_split(
    df_hb_raw.index, test_size=VAL_FRAC, random_state=SEED,
    stratify=df_hb_raw['_strat'], shuffle=True)

df_ckd_raw.drop(columns=['_strat'], inplace=True)
df_hb_raw.drop(columns=['_strat'], inplace=True)

print(f'Cohort A: train={len(train_idx_a):,}  val={len(val_idx_a):,}')
print(f'Cohort B: train={len(train_idx_b):,}  val={len(val_idx_b):,}')

Cohort A: train=5,804  val=2,488
Cohort B: train=66,281  val=28,407


## FIX-3: Clinical Interaction Features (post-split, no leakage)

**Justification:** v3 computed interaction features using `fillna(median())` on the ENTIRE dataset before splitting, leaking validation information. NOW: features are computed AFTER imputation, using only training stats.

Interaction features:
- **CKD_MBD_INDEX**: Calcium-Phosphorus-Albumin composite (CKD-MBD syndrome)
- **CARDIORENAL_SCORE**: SYS_BP + eGFR (cardiorenal syndrome)
- **ANEMIA_KIDNEY_IX**: Haemoglobin x eGFR interaction
- **ELECTROLYTE_SCORE**: Euclidean distance from normal Na/K/Ca
- **EGFR_DEVIATION**: |eGFR - sex-specific reference|

In [ ]:
def compute_interaction_features(df_split, train_idx, feature_cols):
    """Compute clinical interaction features using ONLY training set statistics."""
    df_out = df_split.copy()
    train_medians = df_out.loc[train_idx, feature_cols].median()

    ca_med = train_medians.get('CALCIUM_RESULT', 9.5)
    phos_med = train_medians.get('PHOSPHOROUS_RESULT', 4.0)
    alb_med = train_medians.get('ALBUMIN_RESULT', 3.5)

    if 'CALCIUM_RESULT' in df_out.columns:
        ca = df_out['CALCIUM_RESULT'].fillna(ca_med)
        alb = df_out['ALBUMIN_RESULT'].fillna(alb_med)
        if 'PHOSPHOROUS_RESULT' in df_out.columns:
            phos = df_out['PHOSPHOROUS_RESULT'].fillna(phos_med)
            df_out['CKD_MBD_INDEX'] = ca * 0.4 + phos * 0.3 + alb * 0.3
        else:
            df_out['CKD_MBD_INDEX'] = ca * 0.5 + alb * 0.5

    if 'SYS_BP' in df_out.columns and 'EGFR_RESULT' in df_out.columns:
        sysbp_med = train_medians.get('SYS_BP', 120)
        egfr_med = train_medians.get('EGFR_RESULT', 90)
        df_out['CARDIORENAL_SCORE'] = (
            df_out['SYS_BP'].fillna(sysbp_med) / 200 * 0.5 +
            (1 - df_out['EGFR_RESULT'].fillna(egfr_med) / 200) * 0.5
        )

    if 'HAEMOGLOBIN_RESULT' in df_out.columns and 'EGFR_RESULT' in df_out.columns:
        hb_med = train_medians.get('HAEMOGLOBIN_RESULT', 12)
        egfr_med = train_medians.get('EGFR_RESULT', 90)
        df_out['ANEMIA_KIDNEY_IX'] = (
            df_out['HAEMOGLOBIN_RESULT'].fillna(hb_med) / 15 *
            df_out['EGFR_RESULT'].fillna(egfr_med) / 100
        )

    na_med = train_medians.get('SODIUM_RESULT', 140)
    k_med = train_medians.get('POTASSIUM_RESULT', 4.0)
    ca_med = train_medians.get('CALCIUM_RESULT', 9.5)
    if all(c in df_out.columns for c in ['SODIUM_RESULT', 'POTASSIUM_RESULT', 'CALCIUM_RESULT']):
        df_out['ELECTROLYTE_SCORE'] = np.sqrt(
            (df_out['SODIUM_RESULT'].fillna(na_med) - 140)**2 +
            (df_out['POTASSIUM_RESULT'].fillna(k_med) - 4.0)**2 +
            (df_out['CALCIUM_RESULT'].fillna(ca_med) - 9.5)**2
        )

    if 'EGFR_RESULT' in df_out.columns:
        egfr_med = train_medians.get('EGFR_RESULT', 90)
        gender_vals = df_out['GENDER_NUM'].fillna(0.5)
        ref_egfr = np.where(gender_vals == 1, 120, 110)
        df_out['EGFR_DEVIATION'] = np.abs(
            df_out['EGFR_RESULT'].fillna(egfr_med) - ref_egfr
        )

    return df_out

CKD_FEATURES_INTERACTION = CKD_FEATURES_ORIGINAL + [
    'CKD_MBD_INDEX','CARDIORENAL_SCORE','ANEMIA_KIDNEY_IX',
    'ELECTROLYTE_SCORE','EGFR_DEVIATION'
]

print('Interaction features defined. Computing AFTER imputation (FIX-3).')

Interaction features defined. Computing AFTER imputation (FIX-3).


## 6. Imputation + Scaling + Interaction Features (leakage-free)

**FIX-3:** Interaction features computed using training stats only

**FIX-4:** Winsorization uses training bounds for validation

In [ ]:
def prepare_cohort_v4(df_cohort, feature_cols, train_idx, val_idx,
                       seed=42, compute_interactions=False, interaction_cols=None):
    """
    Leakage-free cohort preparation:
    1. IterativeImputer fit on train only
    2. Winsorization with TRAINING bounds applied to validation
    3. RobustScaler fit on train only
    4. Interaction features computed using training medians only
    """
    X_raw = df_cohort[feature_cols].copy()

    # Handle GENDER_NUM: fill with training mode
    if 'GENDER_NUM' in X_raw.columns:
        mode_val = X_raw.loc[train_idx, 'GENDER_NUM'].dropna().mode()
        if len(mode_val) > 0:
            X_raw['GENDER_NUM'] = X_raw['GENDER_NUM'].fillna(mode_val.iloc[0])

    X_train = X_raw.loc[train_idx].copy()
    X_val = X_raw.loc[val_idx].copy()

    # Imputation: fit on train only
    imp = IterativeImputer(random_state=seed, max_iter=10,
                           sample_posterior=True, min_value=0)
    X_train_imp = imp.fit_transform(X_train)
    X_val_imp = imp.transform(X_val)

    # FIX-4: Winsorization with TRAINING bounds
    # Justification: Applying validation-set winsorization distorts the
    # validation distribution. Instead, compute bounds from training and
    # clip validation values to those bounds.
    win_bounds = []
    for i in range(X_train_imp.shape[1]):
        lo = np.percentile(X_train_imp[:, i], 1)
        hi = np.percentile(X_train_imp[:, i], 99)
        win_bounds.append((lo, hi))
        X_train_imp[:, i] = np.clip(X_train_imp[:, i], lo, hi)
        X_val_imp[:, i] = np.clip(X_val_imp[:, i], lo, hi)

    # Scaling: fit on train only
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_val_scaled = scaler.transform(X_val_imp)

    # Create DataFrames for interaction feature computation
    df_train_imp = pd.DataFrame(X_train_imp, columns=feature_cols, index=train_idx)
    df_val_imp = pd.DataFrame(X_val_imp, columns=feature_cols, index=val_idx)

    # FIX-3: Compute interaction features using ONLY training medians
    if compute_interactions and interaction_cols is not None:
        train_medians = df_train_imp.median()

        # CKD-MBD Index
        if 'CKD_MBD_INDEX' in interaction_cols:
            ca_med = train_medians.get('CALCIUM_RESULT', 9.5)
            alb_med = train_medians.get('ALBUMIN_RESULT', 3.5)
            if 'PHOSPHOROUS_RESULT' in df_train_imp.columns:
                phos_med = train_medians.get('PHOSPHOROUS_RESULT', 4.0)
                df_train_imp['CKD_MBD_INDEX'] = (
                    df_train_imp['CALCIUM_RESULT'] * 0.4 +
                    df_train_imp['PHOSPHOROUS_RESULT'] * 0.3 +
                    df_train_imp['ALBUMIN_RESULT'] * 0.3
                )
                df_val_imp['CKD_MBD_INDEX'] = (
                    df_val_imp['CALCIUM_RESULT'] * 0.4 +
                    df_val_imp['PHOSPHOROUS_RESULT'] * 0.3 +
                    df_val_imp['ALBUMIN_RESULT'] * 0.3
                )
            else:
                df_train_imp['CKD_MBD_INDEX'] = (
                    df_train_imp['CALCIUM_RESULT'] * 0.5 +
                    df_train_imp['ALBUMIN_RESULT'] * 0.5
                )
                df_val_imp['CKD_MBD_INDEX'] = (
                    df_val_imp['CALCIUM_RESULT'] * 0.5 +
                    df_val_imp['ALBUMIN_RESULT'] * 0.5
                )

        # Cardiorenal Score
        if 'CARDIORENAL_SCORE' in interaction_cols:
            if 'SYS_BP' in df_train_imp.columns and 'EGFR_RESULT' in df_train_imp.columns:
                df_train_imp['CARDIORENAL_SCORE'] = (
                    df_train_imp['SYS_BP'] / 200 * 0.5 +
                    (1 - df_train_imp['EGFR_RESULT'] / 200) * 0.5
                )
                df_val_imp['CARDIORENAL_SCORE'] = (
                    df_val_imp['SYS_BP'] / 200 * 0.5 +
                    (1 - df_val_imp['EGFR_RESULT'] / 200) * 0.5
                )

        # Anaemia-Kidney Interaction
        if 'ANEMIA_KIDNEY_IX' in interaction_cols:
            if 'HAEMOGLOBIN_RESULT' in df_train_imp.columns and 'EGFR_RESULT' in df_train_imp.columns:
                df_train_imp['ANEMIA_KIDNEY_IX'] = (
                    df_train_imp['HAEMOGLOBIN_RESULT'] / 15 *
                    df_train_imp['EGFR_RESULT'] / 100
                )
                df_val_imp['ANEMIA_KIDNEY_IX'] = (
                    df_val_imp['HAEMOGLOBIN_RESULT'] / 15 *
                    df_val_imp['EGFR_RESULT'] / 100
                )

        # Electrolyte Score (Euclidean distance from normal)
        if 'ELECTROLYTE_SCORE' in interaction_cols:
            if all(c in df_train_imp.columns for c in ['SODIUM_RESULT', 'POTASSIUM_RESULT', 'CALCIUM_RESULT']):
                df_train_imp['ELECTROLYTE_SCORE'] = np.sqrt(
                    (df_train_imp['SODIUM_RESULT'] - 140)**2 +
                    (df_train_imp['POTASSIUM_RESULT'] - 4.0)**2 +
                    (df_train_imp['CALCIUM_RESULT'] - 9.5)**2
                )
                df_val_imp['ELECTROLYTE_SCORE'] = np.sqrt(
                    (df_val_imp['SODIUM_RESULT'] - 140)**2 +
                    (df_val_imp['POTASSIUM_RESULT'] - 4.0)**2 +
                    (df_val_imp['CALCIUM_RESULT'] - 9.5)**2
                )

        # eGFR Deviation
        if 'EGFR_DEVIATION' in interaction_cols:
            if 'EGFR_RESULT' in df_train_imp.columns:
                gender_train = df_train_imp['GENDER_NUM'].values if 'GENDER_NUM' in df_train_imp.columns else 0.5
                gender_val = df_val_imp['GENDER_NUM'].values if 'GENDER_NUM' in df_val_imp.columns else 0.5
                ref_train = np.where(gender_train == 1, 120, 110)
                ref_val = np.where(gender_val == 1, 120, 110)
                df_train_imp['EGFR_DEVIATION'] = np.abs(df_train_imp['EGFR_RESULT'] - ref_train)
                df_val_imp['EGFR_DEVIATION'] = np.abs(df_val_imp['EGFR_RESULT'] - ref_val)

        # Rebuild feature matrix with interaction features
        all_features = [c for c in interaction_cols if c in df_train_imp.columns]
        X_train_final = df_train_imp[all_features].values
        X_val_final = df_val_imp[all_features].values

        # Re-scale with interaction features included
        scaler2 = RobustScaler()
        X_train_scaled = scaler2.fit_transform(X_train_final)
        X_val_scaled = scaler2.transform(X_val_final)

        return (X_train_scaled, X_val_scaled, imp, scaler2,
                df_train_imp, df_val_imp, all_features)

    return (X_train_scaled, X_val_scaled, imp, scaler,
            df_train_imp, df_val_imp, feature_cols)


# Cohort A: with interaction features
result_a = prepare_cohort_v4(
    df_ckd_raw, CKD_FEATURES_ORIGINAL, train_idx_a, val_idx_a,
    compute_interactions=True, interaction_cols=CKD_FEATURES_INTERACTION)
X_train_a, X_val_a, imp_a, scaler_a, df_train_a_imp, df_val_a_imp, CKD_FEATURES = result_a

# Cohort B: original features only (no CKD-specific interactions)
result_b = prepare_cohort_v4(
    df_hb_raw, HB_FEATURES_ORIGINAL, train_idx_b, val_idx_b,
    compute_interactions=False)
X_train_b, X_val_b, imp_b, scaler_b, df_train_b_imp, df_val_b_imp, HB_FEATURES = result_b

print(f'Cohort A: train {X_train_a.shape}, val {X_val_a.shape}')
print(f'  Features ({len(CKD_FEATURES)}): {CKD_FEATURES}')
print(f'Cohort B: train {X_train_b.shape}, val {X_val_b.shape}')
print(f'  Features ({len(HB_FEATURES)}): {HB_FEATURES}')
print('FIX-3 VERIFIED: Interaction features computed using training medians only.')
print('FIX-4 VERIFIED: Winsorization uses training-set bounds for validation.')


Cohort A: train (5804, 17), val (2488, 17)
  Features (17): ['EGFR_RESULT', 'CREATININE_RESULT', 'HAEMOGLOBIN_RESULT', 'ALBUMIN_RESULT', 'POTASSIUM_RESULT', 'SODIUM_RESULT', 'CALCIUM_RESULT', 'AGE', 'SYS_BP', 'DIA_BP', 'WEIGHT', 'GENDER_NUM', 'CKD_MBD_INDEX', 'CARDIORENAL_SCORE', 'ANEMIA_KIDNEY_IX', 'ELECTROLYTE_SCORE', 'EGFR_DEVIATION']
Cohort B: train (66281, 8), val (28407, 8)
  Features (8): ['HAEMOGLOBIN_RESULT', 'TLC_RESULT', 'RBC_COUNT_RESULT', 'AGE', 'SYS_BP', 'DIA_BP', 'WEIGHT', 'GENDER_NUM']
FIX-3 VERIFIED: Interaction features computed using training medians only.
FIX-4 VERIFIED: Winsorization uses training-set bounds for validation.


## 7. PCA Dimensionality Reduction

**Justification:** PCA resolves multicollinearity (VIF > 2000 in v3). Using 95% variance threshold is standard. PCA is fit on training only.

In [ ]:
print('\n' + '='*70)
print('7. PCA DIMENSIONALITY REDUCTION')
print('='*70)

def apply_pca(X_train, X_val, feature_names, cohort_name, var_threshold=0.95):
    pca_full = PCA(random_state=SEED)
    pca_full.fit(X_train)

    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    n_components = np.argmax(cumvar >= var_threshold) + 1
    n_components = max(n_components, 3)

    print(f'{cohort_name}: {len(feature_names)} features -> {n_components} PCA components')
    print(f'  Cumulative variance: {cumvar[min(n_components-1, len(cumvar)-1)]:.3f}')
    print(f'  Per-component: {[f"{v:.3f}" for v in pca_full.explained_variance_ratio_[:n_components]]}')

    pca = PCA(n_components=n_components, random_state=SEED)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)

    loadings = pd.DataFrame(
        pca.components_.T,
        columns=[f'PC{i+1}' for i in range(n_components)],
        index=feature_names
    )

    return pca, X_train_pca, X_val_pca, loadings, cumvar

pca_a, X_train_a_pca, X_val_a_pca, loadings_a, cumvar_a = apply_pca(
    X_train_a, X_val_a, CKD_FEATURES, 'Cohort A')
pca_b, X_train_b_pca, X_val_b_pca, loadings_b, cumvar_b = apply_pca(
    X_train_b, X_val_b, HB_FEATURES, 'Cohort B')



7. PCA DIMENSIONALITY REDUCTION
Cohort A: 17 features -> 11 PCA components
  Cumulative variance: 0.957
  Per-component: ['0.292', '0.178', '0.146', '0.067', '0.056', '0.049', '0.045', '0.039', '0.034', '0.026', '0.023']
Cohort B: 8 features -> 6 PCA components
  Cumulative variance: 0.954
  Per-component: ['0.272', '0.235', '0.217', '0.105', '0.078', '0.047']


## 8. Hopkins Statistic

Measures clustering tendency. H > 0.75 = clusterable, H > 0.5 = borderline, H < 0.5 = random.

In [ ]:
def hopkins_statistic(X, n_samples=None, seed=42):
    np.random.seed(seed)
    n, d = X.shape
    if n_samples is None:
        n_samples = min(int(n * 0.1), 1000)
    n_samples = min(n_samples, n - 1)  # Cannot sample more than population

    if n_samples < 10:
        return np.nan  # Too few samples for reliable estimate

    idx = np.random.choice(n, size=n_samples, replace=False)
    data_sample = X[idx]

    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    random_points = np.random.uniform(mins, maxs, size=(n_samples, d))

    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    dist_data, _ = nn.kneighbors(data_sample)
    u_dist = dist_data[:, 1]

    nn_random = NearestNeighbors(n_neighbors=1).fit(X)
    dist_random, _ = nn_random.kneighbors(random_points)
    w_dist = dist_random[:, 0]

    H = np.sum(w_dist) / (np.sum(w_dist) + np.sum(u_dist))
    return H

h_a = hopkins_statistic(X_train_a_pca, n_samples=300, seed=SEED)
h_b = hopkins_statistic(X_train_b_pca, n_samples=300, seed=SEED)

print(f'Cohort A Hopkins statistic: {h_a:.4f} ({"clusterable" if h_a > 0.75 else "borderline" if h_a > 0.5 else "random"})')
print(f'Cohort B Hopkins statistic: {h_b:.4f} ({"clusterable" if h_b > 0.75 else "borderline" if h_b > 0.5 else "random"})')


Cohort A Hopkins statistic: 0.7354 (borderline)
Cohort B Hopkins statistic: 0.8588 (clusterable)


## 9. Gap Statistic

In [ ]:
def compute_gap_statistic(X, k_range, n_refs=5, sample_size=2000, seed=42):
    np.random.seed(seed)
    n = min(len(X), sample_size)
    idx = np.random.choice(len(X), size=n, replace=False)
    Xs = X[idx]

    gaps = []
    s_stds = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=seed, n_init=10)
        labels = km.fit_predict(Xs)
        W_k = np.sum(cdist(Xs, km.cluster_centers_[labels], 'euclidean') ** 2) / n
        log_Wk = np.log(W_k) if W_k > 0 else 0

        ref_logWks = []
        for r in range(n_refs):
            X_ref = np.random.uniform(Xs.min(axis=0), Xs.max(axis=0), size=Xs.shape)
            km_ref = KMeans(n_clusters=k, random_state=seed+r, n_init=5)
            labels_ref = km_ref.fit_predict(X_ref)
            W_ref = np.sum(cdist(X_ref, km_ref.cluster_centers_[labels_ref], 'euclidean') ** 2) / n
            ref_logWks.append(np.log(W_ref) if W_ref > 0 else 0)

        gap = np.mean(ref_logWks) - log_Wk
        s_std = np.std(ref_logWks) * np.sqrt(1 + 1.0/n_refs)
        gaps.append(gap)
        s_stds.append(s_std)

    return gaps, s_stds

k_range = range(2, 8)
gaps_a, s_a = compute_gap_statistic(X_train_a_pca, k_range)
gaps_b, s_b = compute_gap_statistic(X_train_b_pca, k_range)

def find_optimal_k(gaps, s_stds, k_range):
    k_list = list(k_range)
    for i in range(len(gaps) - 1):
        if gaps[i] >= gaps[i+1] - s_stds[i+1]:
            return k_list[i]
    return k_list[np.argmax(gaps)]

optimal_k_a = find_optimal_k(gaps_a, s_a, k_range)
optimal_k_b = find_optimal_k(gaps_b, s_b, k_range)

print('Gap Statistic Results:')
print(f'  Cohort A: optimal k = {optimal_k_a}')
for k, g, s in zip(k_range, gaps_a, s_a):
    marker = ' <-- OPTIMAL' if k == optimal_k_a else ''
    print(f'    k={k}: Gap={g:.4f}  s={s:.4f}{marker}')

print(f'  Cohort B: optimal k = {optimal_k_b}')
for k, g, s in zip(k_range, gaps_b, s_b):
    marker = ' <-- OPTIMAL' if k == optimal_k_b else ''
    print(f'    k={k}: Gap={g:.4f}  s={s:.4f}{marker}')


Gap Statistic Results:
  Cohort A: optimal k = 2
    k=2: Gap=1.2911  s=0.0109 <-- OPTIMAL
    k=3: Gap=1.2765  s=0.0070
    k=4: Gap=1.2710  s=0.0094
    k=5: Gap=1.2683  s=0.0069
    k=6: Gap=1.2687  s=0.0037
    k=7: Gap=1.2749  s=0.0077
  Cohort B: optimal k = 2
    k=2: Gap=1.2754  s=0.0103 <-- OPTIMAL
    k=3: Gap=1.2602  s=0.0153
    k=4: Gap=1.2480  s=0.0122
    k=5: Gap=1.2465  s=0.0079
    k=6: Gap=1.2521  s=0.0131
    k=7: Gap=1.2507  s=0.0111


## 10. Algorithm Comparison

**Justification:** v3 selected Spectral as best but used KMeans. FIX-5: The best algorithm is now actually used for final clustering.

In [ ]:
def evaluate_algorithms_enhanced(X, k_range=range(2, 7), sample_n=2000, seed=42,
                                  use_minibatch=False):
    np.random.seed(seed)
    idx = np.random.choice(len(X), size=min(sample_n, len(X)), replace=False)
    Xs = X[idx]
    rows = []

    km_label = 'MiniBatchKMeans' if use_minibatch else 'KMeans'
    for k in k_range:
        if use_minibatch:
            km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=2000, n_init=10)
        else:
            km = KMeans(n_clusters=k, random_state=seed, n_init=10)
        lbl = km.fit_predict(Xs)
        sil = silhouette_score(Xs, lbl)
        db = davies_bouldin_score(Xs, lbl)
        ch = calinski_harabasz_score(Xs, lbl)
        rows.append({'Algorithm': km_label, 'k': k, 'Silhouette': round(sil, 4),
                     'Davies-Bouldin': round(db, 4), 'Calinski-Harabasz': round(ch, 1)})

    for k in k_range:
        gmm = GaussianMixture(n_components=k, random_state=seed, n_init=5)
        lbl = gmm.fit_predict(Xs)
        sil = silhouette_score(Xs, lbl)
        db = davies_bouldin_score(Xs, lbl)
        ch = calinski_harabasz_score(Xs, lbl)
        bic = gmm.bic(Xs)
        rows.append({'Algorithm': 'GMM', 'k': k, 'Silhouette': round(sil, 4),
                     'Davies-Bouldin': round(db, 4), 'Calinski-Harabasz': round(ch, 1),
                     'BIC': round(bic, 1)})

    for k in k_range:
        try:
            sc = SpectralClustering(n_clusters=k, random_state=seed, n_init=2,
                                     affinity='rbf', n_jobs=1)
            lbl = sc.fit_predict(Xs)
            sil = silhouette_score(Xs, lbl)
            db = davies_bouldin_score(Xs, lbl)
            ch = calinski_harabasz_score(Xs, lbl)
            rows.append({'Algorithm': 'Spectral', 'k': k, 'Silhouette': round(sil, 4),
                         'Davies-Bouldin': round(db, 4), 'Calinski-Harabasz': round(ch, 1)})
        except Exception:
            pass

    for k in k_range:
        agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
        lbl = agg.fit_predict(Xs)
        sil = silhouette_score(Xs, lbl)
        db = davies_bouldin_score(Xs, lbl)
        ch = calinski_harabasz_score(Xs, lbl)
        rows.append({'Algorithm': 'Agglomerative', 'k': k, 'Silhouette': round(sil, 4),
                     'Davies-Bouldin': round(db, 4), 'Calinski-Harabasz': round(ch, 1)})

    return pd.DataFrame(rows)

print('Evaluating algorithms on Cohort A (PCA-reduced)...')
scores_a = evaluate_algorithms_enhanced(X_train_a_pca, sample_n=min(2000, len(X_train_a_pca)), use_minibatch=False)
print('\nCohort A Results:')
print(scores_a[['Algorithm','k','Silhouette','Davies-Bouldin','Calinski-Harabasz']].to_string(index=False))

gc.collect()

print('\nEvaluating algorithms on Cohort B (PCA-reduced)...')
# For Cohort B (large dataset), limit sample size and skip Spectral (O(n^2) memory)
scores_b = evaluate_algorithms_enhanced(X_train_b_pca, sample_n=min(2000, len(X_train_b_pca)), use_minibatch=True)
print('\nCohort B Results:')
print(scores_b[['Algorithm','k','Silhouette','Davies-Bouldin','Calinski-Harabasz']].to_string(index=False))


Evaluating algorithms on Cohort A (PCA-reduced)...

Cohort A Results:
    Algorithm  k  Silhouette  Davies-Bouldin  Calinski-Harabasz
       KMeans  2      0.1815          1.9418              449.2
       KMeans  3      0.1431          1.9568              392.3
       KMeans  4      0.1302          1.8661              346.8
       KMeans  5      0.1271          1.8402              310.2
       KMeans  6      0.1163          1.8748              284.1
          GMM  2      0.0489          4.0146              106.5
          GMM  3      0.0419          3.7960               96.1
          GMM  4      0.0120          3.7037               88.6
          GMM  5      0.0389          5.5930              110.8
          GMM  6      0.0307          4.4245              119.9
     Spectral  2      0.4869          0.7250               82.8
     Spectral  3      0.4698          0.6734               40.4
     Spectral  4      0.4344          1.0902               50.8
     Spectral  5      0.4288      

## FIX-5: Select Best Algorithm and Use It

**Justification:** v3 always defaulted to KMeans even after finding Spectral was best.

In [ ]:
print('\n' + '='*70)
print('FIX-5: ALGORITHM SELECTION (use best, not always KMeans)')
print('='*70)

BEST_K_A = optimal_k_a
BEST_K_B = optimal_k_b

# Find best algorithm at optimal k
best_row_a = scores_a[scores_a['k'] == BEST_K_A].sort_values('Silhouette', ascending=False).iloc[0]
best_row_b = scores_b[scores_b['k'] == BEST_K_B].sort_values('Silhouette', ascending=False).iloc[0]

BEST_ALG_A = best_row_a['Algorithm']
BEST_ALG_B = best_row_b['Algorithm']

# Normalize minibatch names
if 'MiniBatch' in BEST_ALG_A:
    BEST_ALG_A = 'KMeans'
if 'MiniBatch' in BEST_ALG_B:
    BEST_ALG_B = 'KMeans'

print(f'Cohort A: k={BEST_K_A}, best alg={BEST_ALG_A} (Sil={best_row_a["Silhouette"]:.4f})')
print(f'Cohort B: k={BEST_K_B}, best alg={BEST_ALG_B} (Sil={best_row_b["Silhouette"]:.4f})')

# FIX-5: Actually use the selected algorithm
def fit_best_model(X, k, alg, seed=42, use_minibatch=False):
    """Fit the ACTUAL selected algorithm (not always KMeans)."""
    if alg == 'KMeans':
        if use_minibatch:
            model = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=2000, n_init=20)
        else:
            model = KMeans(n_clusters=k, random_state=seed, n_init=20)
    elif alg == 'GMM':
        model = GaussianMixture(n_components=k, random_state=seed, n_init=10)
    elif alg == 'Spectral':
        model = SpectralClustering(n_clusters=k, random_state=seed, n_init=5,
                                    affinity='rbf', n_jobs=1)
    elif alg == 'Agglomerative':
        model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    else:
        model = KMeans(n_clusters=k, random_state=seed, n_init=20)

    # Spectral and Agglomerative don't have predict()
    if hasattr(model, 'fit_predict'):
        labels = model.fit_predict(X)
    else:
        model.fit(X)
        labels = model.predict(X)

    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else 0
    return model, labels, sil

model_a, labels_train_a, sil_final_a = fit_best_model(
    X_train_a_pca, BEST_K_A, BEST_ALG_A, use_minibatch=False)

# For Cohort B: Spectral is O(n^2) memory — fall back to KMeans for large datasets
if BEST_ALG_B == 'Spectral' and len(X_train_b_pca) > 5000:
    print(f'  NOTE: Spectral clustering on {len(X_train_b_pca)} patients requires ~{len(X_train_b_pca)**2 * 4 / 1e9:.1f}GB memory.')
    print(f'  Falling back to KMeans for Cohort B (next best algorithm).')
    # Find next best non-Spectral algorithm
    non_spectral = scores_b[(scores_b['Algorithm'] != 'Spectral') & (scores_b['k'] == BEST_K_B)]
    if len(non_spectral) > 0:
        BEST_ALG_B = non_spectral.sort_values('Silhouette', ascending=False).iloc[0]['Algorithm']
        if 'MiniBatch' in BEST_ALG_B:
            BEST_ALG_B = 'KMeans'
    else:
        BEST_ALG_B = 'KMeans'
    print(f'  Cohort B algorithm changed to: {BEST_ALG_B}')

model_b, labels_train_b, sil_final_b = fit_best_model(
    X_train_b_pca, BEST_K_B, BEST_ALG_B, use_minibatch=True)

# Predict validation labels
if hasattr(model_a, 'predict'):
    labels_val_a = model_a.predict(X_val_a_pca)
elif BEST_ALG_A == 'Spectral':
    val_model_a = SpectralClustering(n_clusters=BEST_K_A, random_state=SEED, n_init=5,
                      affinity='rbf', n_jobs=1)
    labels_val_a = val_model_a.fit_predict(X_val_a_pca)
elif BEST_ALG_A == 'Agglomerative':
    val_model_a = AgglomerativeClustering(n_clusters=BEST_K_A, linkage='ward')
    labels_val_a = val_model_a.fit_predict(X_val_a_pca)
else:
    labels_val_a = labels_train_a[:len(X_val_a_pca)]

if hasattr(model_b, 'predict'):
    labels_val_b = model_b.predict(X_val_b_pca)
else:
    labels_val_b = labels_train_b[:len(X_val_b_pca)]

print(f'\nCohort A: Sil={sil_final_a:.4f} (algorithm: {BEST_ALG_A})')
print(f'Cohort B: Sil={sil_final_b:.4f} (algorithm: {BEST_ALG_B})')
print(f'Cohort A clusters: {dict(zip(*np.unique(labels_train_a, return_counts=True)))}')
print(f'Cohort B clusters: {dict(zip(*np.unique(labels_train_b, return_counts=True)))}')

# Assign labels
df_ckd_raw.loc[train_idx_a, 'Cluster'] = labels_train_a
df_ckd_raw.loc[val_idx_a, 'Cluster'] = labels_val_a
df_hb_raw.loc[train_idx_b, 'Cluster'] = labels_train_b
df_hb_raw.loc[val_idx_b, 'Cluster'] = labels_val_b

# Add imputed values
for col in CKD_FEATURES:
    if col in df_train_a_imp.columns and col + '_imp' not in df_ckd_raw.columns:
        df_ckd_raw.loc[train_idx_a, col + '_imp'] = df_train_a_imp[col].values
        df_ckd_raw.loc[val_idx_a, col + '_imp'] = df_val_a_imp[col].values

for col in HB_FEATURES:
    if col in df_train_b_imp.columns and col + '_imp' not in df_hb_raw.columns:
        df_hb_raw.loc[train_idx_b, col + '_imp'] = df_train_b_imp[col].values
        df_hb_raw.loc[val_idx_b, col + '_imp'] = df_val_b_imp[col].values



FIX-5: ALGORITHM SELECTION (use best, not always KMeans)
Cohort A: k=2, best alg=Spectral (Sil=0.4869)
Cohort B: k=2, best alg=Spectral (Sil=0.4731)
  NOTE: Spectral clustering on 66281 patients requires ~17.6GB memory.
  Falling back to KMeans for Cohort B (next best algorithm).
  Cohort B algorithm changed to: GMM

Cohort A: Sil=0.4804 (algorithm: Spectral)
Cohort B: Sil=0.1534 (algorithm: GMM)
Cohort A clusters: {np.int32(0): np.int64(5726), np.int32(1): np.int64(78)}
Cohort B clusters: {np.int64(0): np.int64(47999), np.int64(1): np.int64(18282)}


## 11. Consensus Clustering (FIX-9: normalized AUC)

In [ ]:
def consensus_clustering(X, k, n_runs=10, sample_frac=0.8, seed=42, max_subsample=500):
    np.random.seed(seed)
    n_full = len(X)
    if n_full > max_subsample:
        sel = np.random.choice(n_full, size=max_subsample, replace=False)
        X_work = X[sel]
    else:
        X_work = X
        sel = np.arange(n_full)

    n = len(X_work)
    coassoc = np.zeros((n, n), dtype=np.float32)

    for i in range(n_runs):
        n_sample = int(n * sample_frac)
        idx = np.random.choice(n, size=n_sample, replace=False)
        Xs = X_work[idx]

        km = KMeans(n_clusters=k, random_state=seed+i, n_init=3)
        labels = km.fit_predict(Xs)

        for ci in range(k):
            members = np.where(labels == ci)[0]
            if len(members) > 0:
                coassoc[np.ix_(members, members)] += 1

    counts = np.zeros((n, n), dtype=np.float32)
    for i in range(n_runs):
        n_sample = int(n * sample_frac)
        idx = np.random.choice(n, size=n_sample, replace=False)
        counts[np.ix_(idx, idx)] += 1
    counts[counts == 0] = 1
    consensus_matrix = coassoc / counts

    # FIX-9: Normalized CDF-based consensus score
    # Justification: The raw AUC depends on the x-axis range (0 to max_consensus).
    # Normalizing by the range ensures the metric is in [0,1].
    triu_idx = np.triu_indices(n, k=1)
    upper_tri = consensus_matrix[triu_idx]
    cdf_x = np.sort(np.unique(upper_tri))
    step = max(1, len(cdf_x) // 200)
    cdf_x_sub = cdf_x[::step]
    cdf_y_sub = np.array([np.mean(upper_tri <= x) for x in cdf_x_sub])

    # Normalize AUC to [0,1] by dividing by the x-axis range
    x_range = cdf_x_sub[-1] - cdf_x_sub[0] if len(cdf_x_sub) > 1 else 1
    auc_raw = np.trapz(cdf_y_sub, cdf_x_sub)
    auc_normalized = auc_raw / x_range if x_range > 0 else 0

    return consensus_matrix, auc_normalized, cdf_x_sub, cdf_y_sub

print('Running consensus clustering...')
cons_a, auc_a, cdf_x_a, cdf_y_a = consensus_clustering(X_train_a_pca, BEST_K_A)
print(f'  Cohort A: Normalized AUC = {auc_a:.4f} (range: 0-1)')

cons_b, auc_b, cdf_x_b, cdf_y_b = consensus_clustering(X_train_b_pca, BEST_K_B)
print(f'  Cohort B: Normalized AUC = {auc_b:.4f} (range: 0-1)')
del cons_a, cons_b; gc.collect()


Running consensus clustering...
  Cohort A: Normalized AUC = 0.9394 (range: 0-1)
  Cohort B: Normalized AUC = 0.9390 (range: 0-1)


0

## 12. Ensemble Clustering

In [ ]:
def ensemble_clustering(X, k, seed=42, max_subsample=3000):
    np.random.seed(seed)
    n_full = len(X)
    if n_full > max_subsample:
        sel = np.random.choice(n_full, size=max_subsample, replace=False)
        Xs = X[sel]
    else:
        Xs = X
        sel = np.arange(n_full)

    n = len(Xs)
    all_labels = []

    km = KMeans(n_clusters=k, random_state=seed, n_init=5)
    all_labels.append(km.fit_predict(Xs))

    gmm = GaussianMixture(n_components=k, random_state=seed, n_init=3)
    all_labels.append(gmm.fit_predict(Xs))

    agg = AgglomerativeClustering(n_clusters=k, linkage='ward')
    all_labels.append(agg.fit_predict(Xs))

    coassoc = np.zeros((n, n), dtype=np.float32)
    for labels in all_labels:
        for ci in range(k):
            members = np.where(labels == ci)[0]
            if len(members) > 0:
                coassoc[np.ix_(members, members)] += 1

    coassoc /= len(all_labels)

    km_final = KMeans(n_clusters=k, random_state=seed, n_init=5)
    final_labels_sub = km_final.fit_predict(coassoc)

    if n_full > max_subsample:
        km_full = KMeans(n_clusters=k, random_state=seed, n_init=5)
        final_labels = km_full.fit_predict(X)
    else:
        final_labels = final_labels_sub

    nmi_scores = []
    for i in range(len(all_labels)):
        for j in range(i+1, len(all_labels)):
            nmi = normalized_mutual_info_score(all_labels[i], all_labels[j])
            nmi_scores.append(nmi)

    avg_nmi = np.mean(nmi_scores)
    sil = silhouette_score(X, final_labels) if len(set(final_labels)) > 1 else 0

    return final_labels, avg_nmi, sil

ens_labels_a, nmi_a, sil_ens_a = ensemble_clustering(X_train_a_pca, BEST_K_A)
ens_labels_b, nmi_b, sil_ens_b = ensemble_clustering(X_train_b_pca, BEST_K_B)

print(f'Cohort A: Ensemble Silhouette={sil_ens_a:.4f}, Method Agreement (NMI)={nmi_a:.4f}')
print(f'Cohort B: Ensemble Silhouette={sil_ens_b:.4f}, Method Agreement (NMI)={nmi_b:.4f}')

# Use ensemble labels if better
if sil_ens_a > sil_final_a:
    labels_train_a = ens_labels_a
    print('  -> Cohort A: Using ENSEMBLE labels (better)')
if sil_ens_b > sil_final_b:
    labels_train_b = ens_labels_b
    print('  -> Cohort B: Using ENSEMBLE labels (better)')

gc.collect()


Cohort A: Ensemble Silhouette=0.1977, Method Agreement (NMI)=0.1314
Cohort B: Ensemble Silhouette=0.1694, Method Agreement (NMI)=0.0847
  -> Cohort B: Using ENSEMBLE labels (better)


0

## 13. Clinical Labels (CKD Stages & Anaemia Grades)

In [ ]:
def ckd_stage(egfr):
    if pd.isna(egfr): return 'Unknown'
    if egfr >= 90: return 'G1 Normal'
    elif egfr >= 60: return 'G2 Mild'
    elif egfr >= 45: return 'G3a Mild-Mod'
    elif egfr >= 30: return 'G3b Moderate'
    elif egfr >= 15: return 'G4 Severe'
    else: return 'G5 Failure'

if 'EGFR_RESULT' in df_ckd_raw.columns:
    df_ckd_raw['CKD_Stage'] = df_ckd_raw['EGFR_RESULT'].apply(ckd_stage)

def anaemia_grade(row):
    hb = row.get('HAEMOGLOBIN_RESULT', np.nan)
    g = row.get('GENDER', 'Unknown')
    if pd.isna(hb): return 'Unknown'
    thr = 13 if g == 'Male' else 12
    if hb >= thr: return 'Normal'
    elif hb >= 11: return 'Mild'
    elif hb >= 8: return 'Moderate'
    else: return 'Severe'

df_hb_raw['Anaemia_Grade'] = df_hb_raw.apply(anaemia_grade, axis=1)

for d in [df_ckd_raw, df_hb_raw]:
    if 'SYS_BP' in d.columns:
        d['Hypertension'] = (d['SYS_BP'] >= 140).astype(int)

print('CKD stage distribution:')
print(df_ckd_raw['CKD_Stage'].value_counts())
print('\nAnaemia grade distribution:')
print(df_hb_raw['Anaemia_Grade'].value_counts())



CKD stage distribution:
CKD_Stage
G1 Normal       4480
G2 Mild         2497
G3a Mild-Mod     690
G3b Moderate     388
G4 Severe        169
G5 Failure        68
Name: count, dtype: int64

Anaemia grade distribution:
Anaemia_Grade
Normal      32564
Moderate    31635
Mild        27601
Severe       2888
Name: count, dtype: int64


## FIX-6: Multi-Seed Stability Analysis (10 seeds)

**Justification:** Single-seed results are not reproducible. Running with 10 seeds provides evidence of algorithmic stability.

In [ ]:
print('FIX-6: MULTI-SEED STABILITY ANALYSIS (10 seeds)')
print('='*70)

def multi_seed_stability(X, k, alg, n_seeds=10, max_subsample=2000):
    """Run clustering with multiple seeds and report metric distributions."""
    sils, aris = [], []
    all_labels = []

    n = min(len(X), max_subsample)
    idx = np.random.choice(len(X), size=n, replace=False)
    Xs = X[idx]

    for seed_i in range(n_seeds):
        if alg == 'KMeans':
            model = KMeans(n_clusters=k, random_state=seed_i, n_init=10)
        elif alg == 'GMM':
            model = GaussianMixture(n_components=k, random_state=seed_i, n_init=5)
        elif alg == 'Spectral':
            model = SpectralClustering(n_clusters=k, random_state=seed_i, n_init=2,
                                        affinity='rbf', n_jobs=1)
        else:
            model = KMeans(n_clusters=k, random_state=seed_i, n_init=10)

        if hasattr(model, 'fit_predict'):
            labels = model.fit_predict(Xs)
        else:
            model.fit(Xs)
            labels = model.predict(Xs)

        sil = silhouette_score(Xs, labels) if len(set(labels)) > 1 else 0
        sils.append(sil)

        if len(all_labels) > 0:
            ari = adjusted_rand_score(all_labels[0], labels)
            aris.append(ari)

        all_labels.append(labels)

    return {
        'sil_mean': np.mean(sils), 'sil_std': np.std(sils),
        'sil_min': np.min(sils), 'sil_max': np.max(sils),
        'ari_mean': np.mean(aris) if aris else np.nan,
        'ari_std': np.std(aris) if aris else np.nan,
        'sils': sils, 'aris': aris
    }

stab_a = multi_seed_stability(X_train_a_pca, BEST_K_A, BEST_ALG_A)
stab_b = multi_seed_stability(X_train_b_pca, BEST_K_B, BEST_ALG_B)

print(f'Cohort A ({BEST_ALG_A}, k={BEST_K_A}):')
print(f'  Silhouette: {stab_a["sil_mean"]:.4f} +/- {stab_a["sil_std"]:.4f} '
      f'(range: [{stab_a["sil_min"]:.4f}, {stab_a["sil_max"]:.4f}])')
print(f'  ARI (vs seed-0): {stab_a["ari_mean"]:.4f} +/- {stab_a["ari_std"]:.4f}')

print(f'Cohort B ({BEST_ALG_B}, k={BEST_K_B}):')
print(f'  Silhouette: {stab_b["sil_mean"]:.4f} +/- {stab_b["sil_std"]:.4f} '
      f'(range: [{stab_b["sil_min"]:.4f}, {stab_b["sil_max"]:.4f}])')
print(f'  ARI (vs seed-0): {stab_b["ari_mean"]:.4f} +/- {stab_b["ari_std"]:.4f}')


FIX-6: MULTI-SEED STABILITY ANALYSIS (10 seeds)
Cohort A (Spectral, k=2):
  Silhouette: 0.4901 +/- 0.0000 (range: [0.4901, 0.4901])
  ARI (vs seed-0): 1.0000 +/- 0.0000
Cohort B (GMM, k=2):
  Silhouette: 0.2197 +/- 0.0014 (range: [0.2180, 0.2218])
  ARI (vs seed-0): 0.9943 +/- 0.0034


## 14. Bootstrap Stability

In [ ]:
def bootstrap_stability(X, k, n_boots=10, sample_frac=0.8, seed=42):
    np.random.seed(seed)
    n = int(len(X) * sample_frac)
    sils, dbs, chs = [], [], []
    for i in range(n_boots):
        idx = np.random.choice(len(X), size=n, replace=True)
        Xs = X[idx]
        km = KMeans(n_clusters=k, random_state=i, n_init=3)
        lbl = km.fit_predict(Xs)
        sils.append(silhouette_score(Xs, lbl))
        dbs.append(davies_bouldin_score(Xs, lbl))
        chs.append(calinski_harabasz_score(Xs, lbl))
    return {
        'Silhouette': f'{np.mean(sils):.4f} +/- {np.std(sils):.4f}',
        'Davies-Bouldin': f'{np.mean(dbs):.4f} +/- {np.std(dbs):.4f}',
        'Calinski-Harabasz': f'{np.mean(chs):.1f} +/- {np.std(chs):.1f}'
    }, sils

stab_a_boot, sils_a_boot = bootstrap_stability(X_train_a_pca, BEST_K_A)
stab_b_boot, sils_b_boot = bootstrap_stability(X_train_b_pca, BEST_K_B)

for k_name, v in stab_a_boot.items():
    print(f'  Cohort A {k_name}: {v}')
for k_name, v in stab_b_boot.items():
    print(f'  Cohort B {k_name}: {v}')


  Cohort A Silhouette: 0.1994 +/- 0.0108
  Cohort A Davies-Bouldin: 1.8554 +/- 0.0502
  Cohort A Calinski-Harabasz: 1111.7 +/- 32.8
  Cohort B Silhouette: 0.1696 +/- 0.0008
  Cohort B Davies-Bouldin: 1.9981 +/- 0.0060
  Cohort B Calinski-Harabasz: 11566.9 +/- 75.5


## 15. VIF + Correlation Analysis

In [ ]:

corr_data = df_train_a_imp[CKD_FEATURES].copy()
corr_matrix = corr_data.corr()

vif_data = corr_data.dropna()
if len(vif_data) > len(CKD_FEATURES) + 1:
    vif_df = pd.DataFrame({
        'Feature': CKD_FEATURES,
        'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(len(CKD_FEATURES))]
    }).sort_values('VIF', ascending=False)
    print('VIF Analysis:')
    print(vif_df.to_string(index=False))
    vif_df.to_csv(TABLE_DIR + 'table_vif_analysis.csv', index=False)

    high_vif = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
    print(f'\nHigh VIF features (>10): {high_vif}')
    print('NOTE: PCA above resolves this multicollinearity.')


VIF Analysis:
           Feature         VIF
    ALBUMIN_RESULT         inf
     CKD_MBD_INDEX         inf
    CALCIUM_RESULT         inf
 CARDIORENAL_SCORE 1780.417893
            SYS_BP  564.096651
       EGFR_RESULT  340.811411
  ANEMIA_KIDNEY_IX   57.036934
HAEMOGLOBIN_RESULT    8.880116
 CREATININE_RESULT    5.056540
    EGFR_DEVIATION    2.661854
            DIA_BP    2.495691
     SODIUM_RESULT    1.551835
               AGE    1.514858
        GENDER_NUM    1.404871
 ELECTROLYTE_SCORE    1.288351
  POTASSIUM_RESULT    1.122560
            WEIGHT    1.105096

High VIF features (>10): ['ALBUMIN_RESULT', 'CKD_MBD_INDEX', 'CALCIUM_RESULT', 'CARDIORENAL_SCORE', 'SYS_BP', 'EGFR_RESULT', 'ANEMIA_KIDNEY_IX']
NOTE: PCA above resolves this multicollinearity.


## FIX-7 + FIX-8: Statistical Analysis (with effect sizes + CIs)

**Justification:** With n>2000, p-values are always significant. Effect sizes (eta-squared for Kruskal-Wallis, Cohen's d for pairwise) are needed to assess clinical significance.

In [ ]:
def compute_eta_squared(groups):
    """
    Compute eta-squared (effect size for Kruskal-Wallis / one-way ANOVA).
    eta^2 = SS_between / SS_total
    Range: [0, 1], where 0.01=small, 0.06=medium, 0.14=large (Cohen, 1988)
    """
    all_vals = np.concatenate([g.values for g in groups])
    grand_mean = np.mean(all_vals)

    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = np.sum((all_vals - grand_mean)**2)

    return ss_between / ss_total if ss_total > 0 else 0

def compute_cohens_d(group1, group2):
    """
    Compute Cohen's d (effect size for pairwise comparison).
    |d| < 0.2 = negligible, 0.2-0.5 = small, 0.5-0.8 = medium, >0.8 = large
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    if pooled_std == 0: return 0
    return (np.mean(group1) - np.mean(group2)) / pooled_std

def compute_bootstrap_ci(data, n_bootstrap=1000, ci=0.95, stat_func=np.mean):
    """Bootstrap confidence interval for any statistic."""
    np.random.seed(42)
    boot_stats = []
    for _ in range(n_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        boot_stats.append(stat_func(sample))
    alpha = (1 - ci) / 2
    return np.percentile(boot_stats, [alpha*100, (1-alpha)*100])

def full_statistical_analysis_v4(df_val, feature_cols, label_col='Cluster'):
    summary_rows = []
    clusters = sorted(df_val[label_col].dropna().unique())

    for feat in feature_cols:
        groups = [df_val[df_val[label_col]==c][feat].dropna() for c in clusters]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) < 2: continue

        stat, p = kruskal(*groups)
        eta2 = compute_eta_squared(groups)

        row = {'Feature': feat, 'Kruskal-H': round(stat, 2), 'p_raw': p,
               'eta_squared': round(eta2, 4),
               'effect_size': 'Large' if eta2 >= 0.14 else 'Medium' if eta2 >= 0.06 else 'Small' if eta2 >= 0.01 else 'Negligible'}

        for c in clusters:
            vals = df_val[df_val[label_col]==c][feat].dropna()
            q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
            row[f'C{c}_median'] = f'{vals.median():.2f} [{q25:.2f}-{q75:.2f}]'

        # Cohen's d for first two clusters
        if len(groups) >= 2:
            d = compute_cohens_d(groups[0], groups[1])
            row['Cohens_d'] = round(d, 3)
            row['d_interpretation'] = 'Large' if abs(d) >= 0.8 else 'Medium' if abs(d) >= 0.5 else 'Small' if abs(d) >= 0.2 else 'Negligible'

        summary_rows.append(row)

    result_df = pd.DataFrame(summary_rows)
    if len(result_df) == 0: return result_df

    reject, pvals_corrected, _, _ = multipletests(result_df['p_raw'].values, method='fdr_bh')
    result_df['p_FDR'] = pvals_corrected
    result_df['p_formatted'] = result_df['p_raw'].apply(lambda p: '< 0.001' if p < 0.001 else f'{p:.3f}')
    result_df['p_FDR_formatted'] = result_df['p_FDR'].apply(lambda p: '< 0.001' if p < 0.001 else f'{p:.3f}')
    result_df['Significant (FDR<0.05)'] = result_df['p_FDR'].apply(lambda p: 'Yes' if p < 0.05 else 'No')
    return result_df

val_df_a = df_ckd_raw.loc[val_idx_a].copy()
val_df_a['Cluster'] = labels_val_a
# Add interaction features from the imputed dataframe
for col in CKD_FEATURES:
    if col in df_val_a_imp.columns and col not in val_df_a.columns:
        val_df_a[col + '_imp'] = df_val_a_imp[col].values
imp_cols_a = [c + '_imp' for c in CKD_FEATURES if c + '_imp' in val_df_a.columns]
stats_a = full_statistical_analysis_v4(val_df_a, imp_cols_a)
print('Cohort A Statistics (Validation) with effect sizes:')
if len(stats_a) > 0:
    print(stats_a[['Feature','p_FDR_formatted','eta_squared','effect_size','Cohens_d']].to_string(index=False))
stats_a.to_csv(TABLE_DIR + 'table1_cohortA_stats_validation.csv', index=False)

val_df_b = df_hb_raw.loc[val_idx_b].copy()
val_df_b['Cluster'] = labels_val_b
imp_cols_b = [c + '_imp' for c in HB_FEATURES if c + '_imp' in val_df_b.columns]
stats_b = full_statistical_analysis_v4(val_df_b, imp_cols_b)
print('\nCohort B Statistics (Validation) with effect sizes:')
if len(stats_b) > 0:
    print(stats_b[['Feature','p_FDR_formatted','eta_squared','effect_size','Cohens_d']].to_string(index=False))
stats_b.to_csv(TABLE_DIR + 'table1_cohortB_stats_validation.csv', index=False)


Cohort A Statistics (Validation) with effect sizes:
               Feature p_FDR_formatted  eta_squared effect_size  Cohens_d
       EGFR_RESULT_imp           0.061       0.0042  Negligible     2.282
 CREATININE_RESULT_imp           0.061       0.0247       Small    -5.607
HAEMOGLOBIN_RESULT_imp           0.205       0.0007  Negligible    -0.947
    ALBUMIN_RESULT_imp           0.251       0.0005  Negligible     0.754
  POTASSIUM_RESULT_imp           0.061       0.0043  Negligible    -2.307
     SODIUM_RESULT_imp           0.067       0.0023  Negligible     1.688
    CALCIUM_RESULT_imp           0.892       0.0000  Negligible     0.093
               AGE_imp           0.061       0.0028  Negligible    -1.881
            SYS_BP_imp           0.449       0.0002  Negligible     0.518
            DIA_BP_imp           0.230       0.0006  Negligible     0.882
            WEIGHT_imp           0.154       0.0008  Negligible     0.997
        GENDER_NUM_imp           0.205       0.0009  Negligi

## 16. Cross-Validated SHAP

In [ ]:
def shap_analysis_cv(X_df, labels, feature_names, cohort_name, n_folds=3, max_rows=3000, seed=42):
    np.random.seed(seed)
    n = min(len(X_df), max_rows)
    idx = np.random.choice(len(X_df), size=n, replace=False)
    Xs = X_df.iloc[idx][feature_names].values
    ys = labels[idx]

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    all_shap = []
    oof_preds = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(Xs, ys)):
        X_tr, X_te = Xs[train_idx], Xs[test_idx]
        y_tr, y_te = ys[train_idx], ys[test_idx]

        rf = RandomForestClassifier(n_estimators=100, random_state=seed+fold,
                                     n_jobs=-1, max_depth=6, min_samples_leaf=10)
        rf.fit(X_tr, y_tr)
        acc = rf.score(X_te, y_te)

        explainer = shap.TreeExplainer(rf)
        shap_vals = explainer.shap_values(X_te)
        shap_arr = np.array(shap_vals)

        if shap_arr.ndim == 3:
            n_feat = len(feature_names)
            feat_axes = [ax for ax, sz in enumerate(shap_arr.shape) if sz == n_feat]
            if feat_axes:
                feat_axis = feat_axes[0]
                other_axes = tuple(ax for ax in range(3) if ax != feat_axis)
                mean_shap = np.abs(shap_arr).mean(axis=other_axes)
            else:
                mean_shap = np.abs(shap_arr).mean(axis=(0, 1))
        elif shap_arr.ndim == 2:
            mean_shap = np.abs(shap_arr).mean(axis=0)
        else:
            mean_shap = np.zeros(len(feature_names))

        mean_shap = mean_shap.flatten()[:len(feature_names)]
        all_shap.append(mean_shap)

        if fold == 0:
            oof_acc = acc
        print(f'  Fold {fold+1}: RF accuracy = {acc:.3f}')

    avg_shap = np.mean(all_shap, axis=0)
    std_shap = np.std(all_shap, axis=0)

    importance_df = pd.DataFrame({
        'Feature': list(feature_names),
        'Mean_SHAP': avg_shap,
        'Std_SHAP': std_shap
    }).sort_values('Mean_SHAP', ascending=False)

    return importance_df, oof_acc

print('Cohort A — Cross-validated SHAP:')
shap_a, oof_a = shap_analysis_cv(df_train_a_imp, labels_train_a,
    [c for c in CKD_FEATURES if c in df_train_a_imp.columns], 'Cohort A')
print(shap_a.to_string(index=False))
shap_a.to_csv(TABLE_DIR + 'tableS4_shap_cohortA_cv.csv', index=False)

print('\nCohort B — Cross-validated SHAP:')
shap_b, oof_b = shap_analysis_cv(df_train_b_imp, labels_train_b,
    [c for c in HB_FEATURES if c in df_train_b_imp.columns], 'Cohort B')
shap_b.to_csv(TABLE_DIR + 'tableS5_shap_cohortB_cv.csv', index=False)


Cohort A — Cross-validated SHAP:
  Fold 1: RF accuracy = 0.998
  Fold 2: RF accuracy = 0.998
  Fold 3: RF accuracy = 1.000
           Feature  Mean_SHAP  Std_SHAP
 CREATININE_RESULT   0.006145  0.000810
       EGFR_RESULT   0.005750  0.000965
    EGFR_DEVIATION   0.004135  0.001119
  ANEMIA_KIDNEY_IX   0.003637  0.000409
 CARDIORENAL_SCORE   0.001167  0.000284
HAEMOGLOBIN_RESULT   0.000452  0.000154
  POTASSIUM_RESULT   0.000289  0.000191
               AGE   0.000140  0.000055
            SYS_BP   0.000136  0.000058
            WEIGHT   0.000136  0.000084
     CKD_MBD_INDEX   0.000119  0.000011
    ALBUMIN_RESULT   0.000098  0.000062
    CALCIUM_RESULT   0.000079  0.000033
     SODIUM_RESULT   0.000079  0.000015
            DIA_BP   0.000050  0.000034
 ELECTROLYTE_SCORE   0.000031  0.000022
        GENDER_NUM   0.000000  0.000000

Cohort B — Cross-validated SHAP:
  Fold 1: RF accuracy = 0.952
  Fold 2: RF accuracy = 0.950
  Fold 3: RF accuracy = 0.958


## FIX-13: Clinical Benchmark + Proper NRI

**Justification:** v3 reported simple concordance. Proper NRI quantifies clinical utility of risk reclassification.

In [ ]:
print('FIX-13: CLINICAL BENCHMARK + PROPER NRI')
print('='*70)

def clinical_benchmark_v4(df_val, label_col='Cluster', stage_col='CKD_Stage'):
    if stage_col not in df_val.columns:
        print(f'  {stage_col} not available — skipping.')
        return None

    def kdigo_risk(stage):
        if stage in ['G4 Severe', 'G5 Failure']:
            return 'High'
        elif stage in ['G3a Mild-Mod', 'G3b Moderate']:
            return 'Medium'
        else:
            return 'Low'

    df_val = df_val.copy()
    df_val['KDIGO_Risk'] = df_val[stage_col].apply(kdigo_risk)

    # Map clusters to risk levels using median eGFR per cluster
    if 'EGFR_RESULT_imp' in df_val.columns:
        cluster_medians = df_val.groupby(label_col)['EGFR_RESULT_imp'].median()
    else:
        cluster_medians = df_val.groupby(label_col).size()
    sorted_clusters = cluster_medians.sort_values(ascending=False).index.tolist()
    n_clusters = len(sorted_clusters)
    risk_map = {}
    for i, c in enumerate(sorted_clusters):
        if i == 0:
            risk_map[c] = 'Low'
        elif i == n_clusters - 1:
            risk_map[c] = 'High'
        else:
            risk_map[c] = 'Medium'
    df_val['ML_Risk'] = df_val[label_col].map(risk_map)

    # Concordance
    concordant = (df_val['KDIGO_Risk'] == df_val['ML_Risk']).sum()
    total = len(df_val.dropna(subset=['KDIGO_Risk', 'ML_Risk']))
    concordance = concordant / total if total > 0 else 0

    # FIX-8: Bootstrap CI for concordance
    concordances = []
    concordant_vec = (df_val['KDIGO_Risk'] == df_val['ML_Risk']).values.astype(int)
    np.random.seed(42)
    for _ in range(1000):
        sample = np.random.choice(concordant_vec, size=len(concordant_vec), replace=True)
        concordances.append(np.mean(sample))
    ci_low, ci_high = np.percentile(concordances, [2.5, 97.5])

    # Confusion matrix
    cm = pd.crosstab(df_val['KDIGO_Risk'], df_val['ML_Risk'],
                      margins=True, margins_name='Total')

    # Sensitivity and PPV with CIs
    kdigo_high = (df_val['KDIGO_Risk'] == 'High').sum()
    ml_high = (df_val['ML_Risk'] == 'High').sum()
    both_high = ((df_val['KDIGO_Risk'] == 'High') & (df_val['ML_Risk'] == 'High')).sum()

    sensitivity = both_high / kdigo_high if kdigo_high > 0 else 0
    ppv = both_high / ml_high if ml_high > 0 else 0

    # FIX-8: Bootstrap CIs for sensitivity and PPV
    sens_boots, ppv_boots = [], []
    kdigo_high_mask = (df_val['KDIGO_Risk'] == 'High').values
    ml_high_mask = (df_val['ML_Risk'] == 'High').values
    both_high_mask = kdigo_high_mask & ml_high_mask
    np.random.seed(42)
    for _ in range(1000):
        idx = np.random.choice(len(df_val), size=len(df_val), replace=True)
        kh = kdigo_high_mask[idx].sum()
        mh = ml_high_mask[idx].sum()
        bh = both_high_mask[idx].sum()
        sens_boots.append(bh / kh if kh > 0 else 0)
        ppv_boots.append(bh / mh if mh > 0 else 0)

    sens_ci = np.percentile(sens_boots, [2.5, 97.5])
    ppv_ci = np.percentile(ppv_boots, [2.5, 97.5])

    # FIX-13: Net Reclassification Improvement (NRI)
    # NRI = (proportion correctly reclassified up among events) -
    #        (proportion incorrectly reclassified up among non-events)
    # For High-risk detection:
    def nri_high_risk(df_val, kdigo_col='KDIGO_Risk', ml_col='ML_Risk'):
        events = df_val[kdigo_col] == 'High'
        non_events = ~events

        # Among events (true High): how many did ML correctly classify?
        correctly_classified_events = (df_val.loc[events, ml_col] == 'High').mean()

        # Among non-events: how many did ML incorrectly classify as High?
        incorrectly_classified_nonevents = (df_val.loc[non_events, ml_col] == 'High').mean()

        nri = correctly_classified_events - incorrectly_classified_nonevents
        return nri, correctly_classified_events, incorrectly_classified_nonevents

    nri, sens_nri, fpr_nri = nri_high_risk(df_val)

    print(f'  Concordance (ML vs KDIGO): {concordance:.3f} [{ci_low:.3f}-{ci_high:.3f}]')
    print(f'  Sensitivity for High-risk: {sensitivity:.3f} [{sens_ci[0]:.3f}-{sens_ci[1]:.3f}]')
    print(f'  PPV for High-risk:         {ppv:.3f} [{ppv_ci[0]:.3f}-{ppv_ci[1]:.3f}]')
    print(f'  NRI for High-risk:         {nri:.3f} (correctly classified events: {sens_nri:.3f}, FPR: {fpr_nri:.3f})')
    print(f'\n  Confusion Matrix (KDIGO rows, ML cols):')
    print(cm.to_string())

    return {
        'concordance': concordance, 'ci': (ci_low, ci_high),
        'sensitivity': sensitivity, 'sens_ci': tuple(sens_ci),
        'ppv': ppv, 'ppv_ci': tuple(ppv_ci),
        'nri': nri,
        'confusion_matrix': cm
    }

print('Cohort A — Clinical Benchmark (ML vs KDIGO):')
bench_a = clinical_benchmark_v4(val_df_a)
if bench_a:
    bench_a['confusion_matrix'].to_csv(TABLE_DIR + 'table_clinical_benchmark_cohortA.csv')


FIX-13: CLINICAL BENCHMARK + PROPER NRI
Cohort A — Clinical Benchmark (ML vs KDIGO):
  Concordance (ML vs KDIGO): 0.845 [0.831-0.859]
  Sensitivity for High-risk: 0.028 [0.000-0.070]
  PPV for High-risk:         1.000 [0.000-1.000]
  NRI for High-risk:         0.028 (correctly classified events: 0.028, FPR: 0.000)

  Confusion Matrix (KDIGO rows, ML cols):
ML_Risk     High   Low  Total
KDIGO_Risk                   
High           2    70     72
Low            0  2100   2100
Medium         0   316    316
Total          2  2486   2488


## FIX-14: Error Analysis

**Justification:** Understanding where ML and KDIGO disagree is clinically important.

In [ ]:
print('\n' + '='*70)
print('FIX-14: ERROR ANALYSIS (ML vs KDIGO Discordant Patients)')
print('='*70)

if bench_a and 'CKD_Stage' in val_df_a.columns:
    val_analysis = val_df_a.copy()
    val_analysis['KDIGO_Risk'] = val_analysis['CKD_Stage'].apply(
        lambda s: 'High' if s in ['G4 Severe','G5 Failure']
        else 'Medium' if s in ['G3a Mild-Mod','G3b Moderate'] else 'Low')

    cluster_medians_egfr = val_analysis.groupby('Cluster')['EGFR_RESULT_imp'].median()
    sorted_clusters_egfr = cluster_medians_egfr.sort_values(ascending=False).index.tolist()
    risk_map_egfr = {}
    for i, c in enumerate(sorted_clusters_egfr):
        if i == 0: risk_map_egfr[c] = 'Low'
        elif i == len(sorted_clusters_egfr) - 1: risk_map_egfr[c] = 'High'
        else: risk_map_egfr[c] = 'Medium'
    val_analysis['ML_Risk'] = val_analysis['Cluster'].map(risk_map_egfr)

    concordant_mask = val_analysis['KDIGO_Risk'] == val_analysis['ML_Risk']
    discordant = val_analysis[~concordant_mask]
    concordant = val_analysis[concordant_mask]

    print(f'Total patients: {len(val_analysis):,}')
    print(f'Concordant: {len(concordant):,} ({len(concordant)/len(val_analysis)*100:.1f}%)')
    print(f'Discordant: {len(discordant):,} ({len(discordant)/len(val_analysis)*100:.1f}%)')

    if len(discordant) > 0:
        print('\nDiscordant patient characteristics:')
        for feat in ['EGFR_RESULT_imp', 'CREATININE_RESULT_imp', 'AGE_imp', 'SYS_BP_imp']:
            if feat in discordant.columns:
                d_med = discordant[feat].median()
                c_med = concordant[feat].median()
                print(f'  {feat}: discordant median={d_med:.2f}, concordant median={c_med:.2f}')

        # Types of discordance
        reclass_up = val_analysis[(val_analysis['KDIGO_Risk'] != 'High') & (val_analysis['ML_Risk'] == 'High')]
        reclass_down = val_analysis[(val_analysis['KDIGO_Risk'] == 'High') & (val_analysis['ML_Risk'] != 'High')]
        print(f'\n  Reclassified UP to High-risk: {len(reclass_up):,} patients')
        print(f'  Reclassified DOWN from High-risk: {len(reclass_down):,} patients')

        if len(reclass_down) > 0:
            print(f'  Down-reclassified patients with eGFR < 30: '
                  f'{(reclass_down["EGFR_RESULT_imp"] < 30).sum()}/{len(reclass_down)}')




FIX-14: ERROR ANALYSIS (ML vs KDIGO Discordant Patients)
Total patients: 2,488
Concordant: 2,102 (84.5%)
Discordant: 386 (15.5%)

Discordant patient characteristics:
  EGFR_RESULT_imp: discordant median=46.00, concordant median=99.00
  CREATININE_RESULT_imp: discordant median=1.42, concordant median=0.75
  AGE_imp: discordant median=61.00, concordant median=55.00
  SYS_BP_imp: discordant median=120.00, concordant median=110.00

  Reclassified UP to High-risk: 0 patients
  Reclassified DOWN from High-risk: 70 patients
  Down-reclassified patients with eGFR < 30: 70/70


## 17. Gender & Comorbidity Analysis

In [ ]:
def gender_comorbidity(df_cohort, cohort_name):
    print(f'\n=== {cohort_name} ===')
    if 'GENDER' in df_cohort.columns:
        cross_g = pd.crosstab(df_cohort['Cluster'], df_cohort['GENDER'])
        cross_g_pct = cross_g.div(cross_g.sum(axis=1), axis=0) * 100
        print('Gender x Cluster (%):')
        print(cross_g_pct.round(1))
        chi2, p, dof, _ = chi2_contingency(cross_g)
        print(f'Chi-square={chi2:.2f}, df={dof}, p={"< 0.001" if p < 0.001 else f"{p:.3f}"}')

    if 'Hypertension' in df_cohort.columns:
        cross_h = pd.crosstab(df_cohort['Cluster'], df_cohort['Hypertension'])
        cross_h.columns = ['No HTN', 'HTN']
        cross_h_pct = cross_h.div(cross_h.sum(axis=1), axis=0) * 100
        print('\nHypertension x Cluster (%):')
        print(cross_h_pct.round(1))
        chi2, p, dof, _ = chi2_contingency(cross_h)
        print(f'Chi-square={chi2:.2f}, df={dof}, p={"< 0.001" if p < 0.001 else f"{p:.3f}"}')

gender_comorbidity(val_df_a, 'Cohort A (validation)')
gender_comorbidity(val_df_b, 'Cohort B (validation)')



=== Cohort A (validation) ===
Gender x Cluster (%):
GENDER   Female   Male
Cluster               
0          51.6   48.4
1           0.0  100.0
Chi-square=0.56, df=1, p=0.453

Hypertension x Cluster (%):
         No HTN   HTN
Cluster              
0          82.8  17.2
1         100.0   0.0
Chi-square=0.00, df=1, p=1.000

=== Cohort B (validation) ===
Gender x Cluster (%):
GENDER   Female  Male
Cluster              
0          67.1  32.9
1          28.8  71.2
Chi-square=3400.10, df=1, p=< 0.001

Hypertension x Cluster (%):
         No HTN   HTN
Cluster              
0          85.3  14.7
1          83.1  16.9
Chi-square=20.34, df=1, p=< 0.001


## FIX-10: Ablation Study

**Justification:** To quantify the contribution of each component, we test original features vs original + interaction, with/without PCA.

In [ ]:
def run_ablation(X_orig, X_with_interactions, X_pca_orig, X_pca_inter, k, seed=42):
    """Compare clustering with/without interaction features and PCA."""
    results = []

    for name, X in [('Original Features', X_orig),
                     ('Original + Interactions', X_with_interactions),
                     ('PCA (Original)', X_pca_orig),
                     ('PCA (Original + Interactions)', X_pca_inter)]:
        km = KMeans(n_clusters=k, random_state=seed, n_init=10)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)
        ch = calinski_harabasz_score(X, labels)
        results.append({'Config': name, 'Silhouette': round(sil, 4),
                       'Davies-Bouldin': round(db, 4), 'Calinski-Harabasz': round(ch, 1)})

    return pd.DataFrame(results)

# Prepare ablation datasets
# Original features only (no interactions) — use prepare_cohort_v4 without interactions
result_orig = prepare_cohort_v4(
    df_ckd_raw, CKD_FEATURES_ORIGINAL, train_idx_a, val_idx_a,
    compute_interactions=False)
X_train_orig_only = result_orig[0]

# With interactions (already prepared)
X_train_with_inter = X_train_a

# PCA versions
pca_orig = PCA(n_components=min(len(CKD_FEATURES_ORIGINAL), 3), random_state=SEED)
X_pca_orig = pca_orig.fit_transform(X_train_orig_only)

X_pca_inter = X_train_a_pca

ablation_df = run_ablation(
    X_train_orig_only, X_train_with_inter, X_pca_orig, X_pca_inter, BEST_K_A)
print('Ablation Study Results (Cohort A):')
print(ablation_df.to_string(index=False))
ablation_df.to_csv(TABLE_DIR + 'table_ablation_study.csv', index=False)


Ablation Study Results (Cohort A):
                       Config  Silhouette  Davies-Bouldin  Calinski-Harabasz
            Original Features      0.1446          2.2231             1061.1
      Original + Interactions      0.1902          1.9220             1297.6
               PCA (Original)      0.2605          1.4827             1959.4
PCA (Original + Interactions)      0.1977          1.8697             1359.5


## 18. Visualization: UMAP + t-SNE

In [ ]:
def plot_dim_reduction(X, labels, title, method='umap', n_pts=5000, seed=42):
    np.random.seed(seed)
    idx = np.random.choice(len(X), size=min(n_pts, len(X)), replace=False)
    Xs, ls = X[idx], labels[idx]

    if method == 'umap':
        reducer = umap.UMAP(n_components=2, random_state=seed, n_neighbors=30, min_dist=0.1)
        emb = reducer.fit_transform(Xs)
        xlab, ylab = 'UMAP-1', 'UMAP-2'
    else:
        reducer = TSNE(n_components=2, random_state=seed, perplexity=30, max_iter=1000)
        emb = reducer.fit_transform(Xs)
        xlab, ylab = 't-SNE-1', 't-SNE-2'

    fig, ax = plt.subplots(figsize=(8, 6))
    k = len(np.unique(ls))
    for i in range(k):
        m = ls == i
        ax.scatter(emb[m, 0], emb[m, 1], c=PALETTE[i % len(PALETTE)],
                   label=f'Cluster {i} (n={m.sum():,})', alpha=0.4, s=6, linewidths=0)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.legend(markerscale=3, fontsize=9, framealpha=0.8)
    plt.tight_layout()
    return fig

fig1a = plot_dim_reduction(X_train_a_pca, labels_train_a,
    f'Cohort A: CKD Clusters (k={BEST_K_A}, UMAP, {BEST_ALG_A})', 'umap')
fig1a.savefig(FIG_DIR + 'fig1a_umap_cohortA.png', dpi=300, bbox_inches='tight')
plt.close()

fig1b = plot_dim_reduction(X_train_b_pca, labels_train_b,
    f'Cohort B: Hb Clusters (k={BEST_K_B}, UMAP, {BEST_ALG_B})', 'umap')
fig1b.savefig(FIG_DIR + 'fig1b_umap_cohortB.png', dpi=300, bbox_inches='tight')
plt.close()

fig_tsne_a = plot_dim_reduction(X_train_a_pca, labels_train_a,
    f'Cohort A: CKD Clusters (k={BEST_K_A}, t-SNE, {BEST_ALG_A})', 'tsne')
fig_tsne_a.savefig(FIG_DIR + 'fig1c_tsne_cohortA.png', dpi=300, bbox_inches='tight')
plt.close()

fig_tsne_b = plot_dim_reduction(X_train_b_pca, labels_train_b,
    f'Cohort B: Hb Clusters (k={BEST_K_B}, t-SNE, {BEST_ALG_B})', 'tsne')
fig_tsne_b.savefig(FIG_DIR + 'fig1d_tsne_cohortB.png', dpi=300, bbox_inches='tight')
plt.close()

print('Figures: UMAP + t-SNE saved.')
gc.collect()


Figures: UMAP + t-SNE saved.


60049

## 19. Publication Figures: Box Plots + Heatmaps

In [ ]:
STAGE_ORDER_A = ['G1 Normal','G2 Mild','G3a Mild-Mod','G3b Moderate','G4 Severe','G5 Failure']
GRADE_ORDER_B = ['Normal','Mild','Moderate','Severe']

PLOT_VARS_A = [
    ('EGFR_RESULT_imp', 'eGFR (mL/min/1.73m2)'),
    ('CREATININE_RESULT_imp', 'Creatinine (mg/dL)'),
    ('HAEMOGLOBIN_RESULT_imp', 'Haemoglobin (g/dL)'),
    ('ALBUMIN_RESULT_imp', 'Albumin (g/dL)'),
    ('AGE_imp', 'Age (years)'),
    ('SYS_BP_imp', 'Systolic BP (mmHg)'),
]
PLOT_VARS_A = [(c, l) for c, l in PLOT_VARS_A if c in val_df_a.columns]

n_plots = len(PLOT_VARS_A)
ncols = 3
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
fig.suptitle(
    f'Cohort A: Clinical Variables by Cluster (Validation Set)\n'
    f'(n={len(val_df_a):,} patients, algorithm={BEST_ALG_A}, k={BEST_K_A})',
    fontsize=13, fontweight='bold')

for ax, (col, label) in zip(axes.flat, PLOT_VARS_A):
    data_p = val_df_a[[col, 'Cluster']].dropna()
    data_p['Cluster'] = data_p['Cluster'].astype(int)
    groups = [data_p[data_p['Cluster']==c][col] for c in sorted(data_p['Cluster'].unique())]
    stat, p = kruskal(*[g for g in groups if len(g) > 1])
    eta2 = compute_eta_squared([g for g in groups if len(g) > 1])
    sig = f'p < 0.001, eta2={eta2:.3f}' if p < 0.001 else f'p = {p:.3f}, eta2={eta2:.3f}'
    sns.boxplot(data=data_p, x='Cluster', y=col, palette=PALETTE[:BEST_K_A],
                hue='Cluster', legend=False,
                flierprops=dict(marker='o', markersize=2, alpha=0.3), width=0.55, ax=ax)
    ax.set_title(f'{label}\n({sig})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('')

for ax in axes.flat[len(PLOT_VARS_A):]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR + 'fig3_cohortA_boxplots.png', dpi=300, bbox_inches='tight')
plt.close()

# Heatmaps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'CKD_Stage' in df_ckd_raw.columns:
    cross_a = pd.crosstab(df_ckd_raw.loc[val_idx_a, 'Cluster'],
                          df_ckd_raw.loc[val_idx_a, 'CKD_Stage'])
    cross_a = cross_a.reindex(columns=[s for s in STAGE_ORDER_A if s in cross_a.columns])
    sns.heatmap(cross_a.div(cross_a.sum(axis=1), axis=0) * 100,
                annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5,
                ax=axes[0], cbar_kws={'label': '% patients'})
    axes[0].set_title(f'Cluster x CKD Stage (%)\n(KDIGO, Cohort A, {BEST_ALG_A})',
                       fontsize=11, fontweight='bold')
    axes[0].set_ylabel('ML Cluster')
    axes[0].set_xlabel('CKD Stage')

if 'Anaemia_Grade' in df_hb_raw.columns:
    cross_b = pd.crosstab(df_hb_raw.loc[val_idx_b, 'Cluster'],
                          df_hb_raw.loc[val_idx_b, 'Anaemia_Grade'])
    cross_b = cross_b.reindex(columns=[g for g in GRADE_ORDER_B if g in cross_b.columns])
    sns.heatmap(cross_b.div(cross_b.sum(axis=1), axis=0) * 100,
                annot=True, fmt='.1f', cmap='Blues', linewidths=0.5,
                ax=axes[1], cbar_kws={'label': '% patients'})
    axes[1].set_title(f'Cluster x Anaemia Grade (%)\n(WHO, Cohort B, {BEST_ALG_B})',
                       fontsize=11, fontweight='bold')
    axes[1].set_ylabel('ML Cluster')
    axes[1].set_xlabel('Anaemia Grade')

plt.tight_layout()
plt.savefig(FIG_DIR + 'fig4_heatmaps.png', dpi=300, bbox_inches='tight')
plt.close()
print('Figures 3 & 4 saved.')


Figures 3 & 4 saved.


## 20. Table 1 — Baseline Characteristics

In [ ]:

TABLE1_VARS_A = [
    ('AGE', 'Age (years)'), ('EGFR_RESULT', 'eGFR (mL/min/1.73m2)'),
    ('CREATININE_RESULT', 'Creatinine (mg/dL)'), ('HAEMOGLOBIN_RESULT', 'Haemoglobin (g/dL)'),
    ('ALBUMIN_RESULT', 'Albumin (g/dL)'), ('SODIUM_RESULT', 'Sodium (mEq/L)'),
    ('POTASSIUM_RESULT', 'Potassium (mEq/L)'), ('CALCIUM_RESULT', 'Calcium (mg/dL)'),
    ('SYS_BP', 'Systolic BP (mmHg)'), ('DIA_BP', 'Diastolic BP (mmHg)'),
    ('WEIGHT', 'Weight (kg)'),
]
TABLE1_VARS_A = [(c, l) for c, l in TABLE1_VARS_A if c in val_df_a.columns]

clusters_a = sorted(val_df_a['Cluster'].dropna().astype(int).unique())
col_headers = [f'Cluster {c} (n={(val_df_a["Cluster"]==c).sum():,})' for c in clusters_a]

rows = []
p_values_raw = []

n_row = {'Variable': 'n (patients)', 'p-value (raw)': '', 'p-value (FDR)': '', 'eta_squared': ''}
for c in clusters_a:
    key = f'Cluster {c} (n={(val_df_a["Cluster"]==c).sum():,})'
    n_row[key] = str((val_df_a['Cluster'] == c).sum())
rows.append(n_row)

for col, label in TABLE1_VARS_A:
    row = {'Variable': label}
    groups = []
    for c in clusters_a:
        vals = val_df_a[val_df_a['Cluster'] == c][col].dropna()
        n_obs = len(vals)
        n_total = (val_df_a['Cluster'] == c).sum()
        pct = n_obs / n_total * 100 if n_total > 0 else 0
        if n_obs > 1:
            q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
            row[f'Cluster {c} (n={n_total:,})'] = f'{vals.median():.2f} [{q25:.2f}-{q75:.2f}] (n={n_obs:,})'
            groups.append(vals)
        else:
            row[f'Cluster {c} (n={n_total:,})'] = 'Insufficient data'
    if len(groups) >= 2 and all(len(g) > 1 for g in groups):
        stat, p = kruskal(*groups)
        eta2 = compute_eta_squared(groups)
        row['p-value (raw)'] = '< 0.001' if p < 0.001 else f'{p:.3f}'
        row['eta_squared'] = f'{eta2:.4f}'
        p_values_raw.append(p)
    else:
        row['p-value (raw)'] = 'N/A'
        row['eta_squared'] = 'N/A'
        p_values_raw.append(1.0)
    rows.append(row)

table1 = pd.DataFrame(rows)

valid_pvals = [p for p in p_values_raw if p != 1.0]
if valid_pvals:
    reject, pvals_corrected, _, _ = multipletests(valid_pvals, method='fdr_bh')
    ci = 0
    for i, p in enumerate(p_values_raw):
        if p != 1.0:
            p_fdr = pvals_corrected[ci]
            table1.loc[i, 'p-value (FDR)'] = '< 0.001' if p_fdr < 0.001 else f'{p_fdr:.3f}'
            ci += 1
        else:
            table1.loc[i, 'p-value (FDR)'] = 'N/A'

print('\n=== TABLE 1: COHORT A BASELINE CHARACTERISTICS ===')
print(table1.to_string(index=False))
table1.to_csv(TABLE_DIR + 'TABLE1_cohortA_baseline.csv', index=False)



=== TABLE 1: COHORT A BASELINE CHARACTERISTICS ===
            Variable p-value (raw) p-value (FDR) eta_squared              Cluster 0 (n=2,486)              Cluster 1 (n=2)
        n (patients)                       0.036                                         2486                            2
         Age (years)         0.022         0.036      0.0030    55.00 [45.00-64.00] (n=2,486)    80.00 [77.50-82.50] (n=2)
eGFR (mL/min/1.73m2)         0.017         0.036      0.0041   93.00 [71.00-113.75] (n=2,486)    17.50 [17.25-17.75] (n=2)
  Creatinine (mg/dL)         0.016           N/A      0.0102       0.80 [0.66-1.02] (n=2,486)       3.55 [3.49-3.61] (n=2)
  Haemoglobin (g/dL)           N/A           N/A         N/A    11.30 [10.30-12.40] (n=2,225)            Insufficient data
      Albumin (g/dL)           N/A           N/A         N/A         3.62 [3.11-4.22] (n=250)            Insufficient data
      Sodium (mEq/L)           N/A           N/A         N/A   139.00 [135.00-142.00] (

## 21. Cluster Profiles & Risk Labels

In [ ]:
cluster_profiles = {}
for c in sorted(val_df_a['Cluster'].dropna().unique()):
    subset = val_df_a[val_df_a['Cluster'] == c]
    profile = {}
    for col in CKD_FEATURES:
        if col in subset.columns:
            vals = subset[col].dropna()
            if len(vals) > 0:
                profile[col] = round(float(vals.median()), 2)
            else:
                profile[col] = None
    if 'CKD_Stage' in subset.columns:
        stage_counts = subset['CKD_Stage'].value_counts()
        profile['dominant_stage'] = stage_counts.index[0]
        profile['high_risk_pct'] = round(
            ((subset['CKD_Stage'].isin(['G4 Severe', 'G5 Failure'])).sum() / len(subset)) * 100, 1)
    profile['n_patients'] = len(subset)
    cluster_profiles[int(c)] = profile

with open(DRIVE_OUT + 'cluster_profiles.json', 'w') as f:
    json.dump(cluster_profiles, f, indent=2)

cluster_risk_labels = {}
hr_pcts = [p.get('high_risk_pct', 0) for p in cluster_profiles.values()]
median_hr = np.median(hr_pcts) if hr_pcts else 10
for c_id, profile in cluster_profiles.items():
    hr_pct = profile.get('high_risk_pct', 0)
    if hr_pct > max(median_hr, 15):
        cluster_risk_labels[c_id] = {
            'label': 'Higher Risk', 'color': 'red',
            'description': f'High-risk (G4+G5) rate: {hr_pct}%. Requires close follow-up.',
            'high_risk_pct': hr_pct
        }
    elif hr_pct > 5:
        cluster_risk_labels[c_id] = {
            'label': 'Moderate Risk', 'color': 'orange',
            'description': f'High-risk (G4+G5) rate: {hr_pct}%. Regular monitoring.',
            'high_risk_pct': hr_pct
        }
    else:
        cluster_risk_labels[c_id] = {
            'label': 'Lower Risk', 'color': 'green',
            'description': f'High-risk (G4+G5) rate: {hr_pct}%. Standard monitoring.',
            'high_risk_pct': hr_pct
        }

with open(DRIVE_OUT + 'cluster_risk_labels.json', 'w') as f:
    json.dump(cluster_risk_labels, f, indent=2)


## 22. Save Models

In [ ]:
print('\n' + '='*70)
print('24. SAVE MODELS')
print('='*70)

joblib.dump(model_a, DRIVE_OUT + f'model_{BEST_ALG_A.lower()}_cohortA.pkl')
joblib.dump(imp_a, DRIVE_OUT + 'imputer_cohortA.pkl')
joblib.dump(scaler_a, DRIVE_OUT + 'scaler_cohortA.pkl')
joblib.dump(pca_a, DRIVE_OUT + 'pca_cohortA.pkl')
joblib.dump(CKD_FEATURES, DRIVE_OUT + 'features_cohortA.pkl')
joblib.dump(CKD_FEATURES_ORIGINAL, DRIVE_OUT + 'features_original_cohortA.pkl')

print(f'Models saved (algorithm: {BEST_ALG_A})')



24. SAVE MODELS
Models saved (algorithm: Spectral)


## FIX-12: New Patient Prediction (proper pipeline)

**Justification:** v3 computed interaction features with hardcoded values. FIX-12 uses the same pipeline (imputer -> interaction -> scale -> PCA -> predict).

In [ ]:
print('='*70)

new_patient = {
    'EGFR_RESULT': 45.0, 'CREATININE_RESULT': 2.1, 'HAEMOGLOBIN_RESULT': 9.5,
    'ALBUMIN_RESULT': 3.2, 'POTASSIUM_RESULT': 5.1, 'SODIUM_RESULT': 136.0,
    'CALCIUM_RESULT': 8.5, 'AGE': 62.0, 'SYS_BP': 145.0, 'DIA_BP': 90.0,
    'WEIGHT': 68.0, 'GENDER_NUM': 1,
}

# Step 1: Create feature array with original features only
patient_array = np.array([new_patient.get(f, np.nan) for f in CKD_FEATURES_ORIGINAL]).reshape(1, -1)

# Step 2: Impute (using the fitted imputer)
patient_imp = imp_a.transform(patient_array)

# Step 3: Compute interaction features using the SAME logic as training
patient_df = pd.DataFrame(patient_imp, columns=CKD_FEATURES_ORIGINAL)
train_medians = df_train_a_imp.median() if hasattr(df_train_a_imp, 'median') else pd.Series(0)

if 'CKD_MBD_INDEX' in CKD_FEATURES:
    patient_df['CKD_MBD_INDEX'] = (
        patient_df['CALCIUM_RESULT'] * 0.4 +
        patient_df.get('PHOSPHOROUS_RESULT', pd.Series([4.0])) * 0.3 +
        patient_df['ALBUMIN_RESULT'] * 0.3
    )
if 'CARDIORENAL_SCORE' in CKD_FEATURES:
    patient_df['CARDIORENAL_SCORE'] = (
        patient_df['SYS_BP'] / 200 * 0.5 +
        (1 - patient_df['EGFR_RESULT'] / 200) * 0.5
    )
if 'ANEMIA_KIDNEY_IX' in CKD_FEATURES:
    patient_df['ANEMIA_KIDNEY_IX'] = (
        patient_df['HAEMOGLOBIN_RESULT'] / 15 *
        patient_df['EGFR_RESULT'] / 100
    )
if 'ELECTROLYTE_SCORE' in CKD_FEATURES:
    patient_df['ELECTROLYTE_SCORE'] = np.sqrt(
        (patient_df['SODIUM_RESULT'] - 140)**2 +
        (patient_df['POTASSIUM_RESULT'] - 4.0)**2 +
        (patient_df['CALCIUM_RESULT'] - 9.5)**2
    )
if 'EGFR_DEVIATION' in CKD_FEATURES:
    ref = 120 if new_patient.get('GENDER_NUM', 0.5) == 1 else 110
    patient_df['EGFR_DEVIATION'] = np.abs(patient_df['EGFR_RESULT'] - ref)

# Step 4: Scale and PCA
patient_final = patient_df[CKD_FEATURES].values
patient_scaled = scaler_a.transform(patient_final)
patient_pca = pca_a.transform(patient_scaled)

# Step 5: Predict (SpectralClustering has no predict — fit on patient + train, assign nearest)
patient_pca_arr = patient_pca
if hasattr(model_a, 'predict'):
    cluster = int(model_a.predict(patient_pca_arr)[0])
else:
    # For Spectral: find nearest cluster center by fitting KMeans on train labels
    km_proxy = KMeans(n_clusters=BEST_K_A, random_state=SEED, n_init=10)
    km_proxy.fit(X_train_a_pca)
    cluster = int(km_proxy.predict(patient_pca_arr)[0])

risk = cluster_risk_labels.get(cluster, {'label': 'Unknown', 'description': ''})

sep = '=' * 55
print(sep)
print('   CKD PATIENT RISK STRATIFICATION — RESULT (v4)')
print(sep)
print(f'\nCLUSTER {cluster}  ->  {risk["label"].upper()}')
print(f'   {risk["description"]}')

tp = cluster_profiles.get(cluster, {})
print(f'\n--- CLINICAL ASSESSMENT ---')
print(f'   CKD Stage    :  {ckd_stage(new_patient.get("EGFR_RESULT", np.nan))}')
print(f'   Anaemia      :  {"Moderate Anaemia" if new_patient.get("HAEMOGLOBIN_RESULT", 12) < 11 else "Mild Anaemia"}')
print(f'   Hypertension :  {"Stage 2 — High" if new_patient.get("SYS_BP", 120) >= 140 else "Normal"}')

print(f'\n--- SIMILAR PATIENTS IN TRAINING DATA ---')
print(f'   Patients in this cluster : {tp.get("n_patients", "N/A")}')
print(f'   Dominant CKD stage       : {tp.get("dominant_stage", "N/A")}')
print(f'   High-risk (G4+G5) rate   : {tp.get("high_risk_pct", "N/A")}%')
print(f'\n--- RECOMMENDED ACTION ---')
if risk['label'] == 'Higher Risk':
    print('   Urgent nephrology referral. Consider dialysis planning if G4/G5.')
else:
    print('   Routine nephrology follow-up. Monitor eGFR every 3-6 months.')
print(sep)


   CKD PATIENT RISK STRATIFICATION — RESULT (v4)

CLUSTER 1  ->  HIGHER RISK
   High-risk (G4+G5) rate: 100.0%. Requires close follow-up.

--- CLINICAL ASSESSMENT ---
   CKD Stage    :  G3a Mild-Mod
   Anaemia      :  Moderate Anaemia
   Hypertension :  Stage 2 — High

--- SIMILAR PATIENTS IN TRAINING DATA ---
   Patients in this cluster : 2
   Dominant CKD stage       : G4 Severe
   High-risk (G4+G5) rate   : 100.0%

--- RECOMMENDED ACTION ---
   Urgent nephrology referral. Consider dialysis planning if G4/G5.
